In [9]:
# Importando bibliotecas
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from pandas.tseries.offsets import MonthEnd
from functions import *
from FUNCTIONS_LTP import *
import locale
from openpyxl import load_workbook
from openpyxl.utils import range_boundaries
import tempfile
import shutil
import math
from collections import defaultdict
import pyarrow

locale.setlocale(locale.LC_TIME, 'Portuguese_Brazil.1252')  # Para Windows
timer = Temporizador()
timer.iniciar()

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Aumenta o limite de largura da coluna para exibição
pd.set_option('display.expand_frame_repr', False)

# Detecta se o script está sendo executado de um .py ou de um notebook
try:
    caminho_base = Path(__file__).resolve().parent
except NameError:
    # __file__ não existe em Jupyter ou ambiente interativo
    caminho_base = Path.cwd()
    
pasta_input = caminho_base.parent / '01_INPUT'
pasta_output = caminho_base.parent / '02_OUTPUT'
pasta_painel = caminho_base.parent / '03_EXCEL'

# Carregar bd_mat_param uma vez
bd_mat_param = pd.read_excel(
    pasta_input / 'matriz_parametros.xlsx',
    sheet_name='matriz_parametros',
    engine='calamine',
    dtype={'produto': str}
)
bd_mat_param['produto'] = bd_mat_param['produto'].astype(str).str.strip().str.upper()
produtos_param = set(bd_mat_param['produto'])
colunas_manter = [
    'produto', 'unidade_fat', 'unidade_prod', 'tipo_abast', 'prioridade'
]

bd_mat_param = bd_mat_param[colunas_manter].reset_index(drop=True)

# Carregar base_dados_produtos
bd_prod = pd.read_excel(
    pasta_input / 'base_dados_produtos.xlsx',
    sheet_name='base_dados_produtos',
    engine='calamine',
    dtype={'cod_produto': str}
)
bd_prod['cod_produto'] = bd_prod['cod_produto'].astype(str).str.strip().str.upper()
bd_prod['cod_produto'] = bd_prod['cod_produto'].astype(str).str.strip().str.upper()
bd_prod = bd_prod[bd_prod['cod_produto'].isin(produtos_param)].copy().reset_index(drop=True)
if 'mes_ref' in bd_prod.columns:
    bd_prod['mes_ref'] = pd.to_datetime(bd_prod['mes_ref'], errors='coerce', dayfirst=True)

colunas_manter = ['mes_ref', 'empresa', 'cod_produto', 'descricao', 'linha_prod', 'familia_prod', 'tipo_produto', 'curva_abc', 'curva_123', 'estoq_seg_pcs', 'estoq_seg_kg', 'estoq_inicial_pcs', 'estoq_inicial_kg', 'carteira_arraste_mes_anterior', 'carteira_mes_atual', 'previsao_pcs', 'saldo_previsao_pcs', 'peso_produto_kg', 'estoq_transf_pcs', 'lote_econ', 'qtd_emb', 'lote_min']
bd_prod = bd_prod[colunas_manter].reset_index(drop=True)

# Carregar estruturas
colunas_bd_estruturas = ['mes_ref', 'empresa', 'cod_prod_acabado', 'cod_insumo', 'descricao', 'qtd_utilizada_pcs']
bd_estruturas = pd.read_excel(
    pasta_input / 'estruturas.xlsx',
    sheet_name='estruturas',
    engine='calamine',
    dtype={'cod_prod_acabado': str}
)
bd_estruturas['cod_prod_acabado'] = bd_estruturas['cod_prod_acabado'].astype(str).str.strip().str.upper()
bd_estruturas['cod_insumo'] = bd_estruturas['cod_insumo'].astype(str).str.strip().str.upper()
bd_estruturas = bd_estruturas[bd_estruturas['cod_prod_acabado'].isin(produtos_param)].copy().reset_index(drop=True)
bd_estruturas = bd_estruturas[colunas_bd_estruturas]
# Elminar coluna mes_ref e remover duplicatas
bd_estruturas = bd_estruturas.drop(columns=['mes_ref'])
bd_estruturas = bd_estruturas.drop_duplicates(subset=['empresa', 'cod_prod_acabado', 'cod_insumo']).reset_index(drop=True)

print("✅ Mapeamento de pastas e importação de tabelas concluído!")

✅ Mapeamento de pastas e importação de tabelas concluído!


In [10]:
# Gerando a tabela Calendário

# Definir intervalo de datas: do primeiro mês ao último mês completo
data_inicial = bd_prod['mes_ref'].min().replace(day=1)
data_final = bd_prod['mes_ref'].max() + MonthEnd(1)

# Gerar datas diariamente entre data_inicial e data_final
datas = pd.date_range(start=data_inicial, end=data_final, freq='D')

# Mapeamento dos dias da semana
dias_semana = {0: 'SEG', 1: 'TER', 2: 'QUA', 3: 'QUI', 4: 'SEX', 5: 'SÁB', 6: 'DOM'}

# Construção do DataFrame
df_calendario = pd.DataFrame({
    'data_calend': datas,
    'mes_calend': datas.to_series().apply(lambda d: d.replace(day=1)),
    'dia_calend': datas.to_series().dt.weekday.map(dias_semana)
})

# Importar aba Dia_Semana da planilha dados_calendario, com dados por dia da semana x Unidade
df_dia_semana = pd.read_excel(pasta_input / 'dados_calendario.xlsx', engine='calamine', sheet_name='Dia_Semana')

# Normalização (unpivot)
bd_dia_semana = df_dia_semana.melt(
    id_vars=['UNIDADE'],
    var_name='DIA_DE_SEMANA',
    value_name='OCUPACAO'
)

# Remove linhas onde OCUPACAO está vazia ou inválida (ex: '-')
bd_dia_semana = bd_dia_semana[bd_dia_semana['OCUPACAO'].notna()]
bd_dia_semana = bd_dia_semana[bd_dia_semana['OCUPACAO'] != '-']
bd_dia_semana['OCUPACAO'] = bd_dia_semana['OCUPACAO'].astype(float)

# Fazendo o merge entre o calendário e os dados de ocupação por unidade/dia da semana
df_calendario_ocup = df_calendario.merge(
    bd_dia_semana,
    left_on='dia_calend',
    right_on='DIA_DE_SEMANA',
    how='left'
)

# Selecionar apenas as colunas desejadas
df_calendario_ocup = df_calendario_ocup[
    ['data_calend', 'mes_calend', 'dia_calend', 'UNIDADE', 'OCUPACAO']
]

# Renomear colunas para maiúsculas
df_calendario_ocup.columns = [col.upper() for col in df_calendario_ocup.columns]

# Importar aba Feriados da planilha dados_calendario, com dados por dia da semana x Unidade
df_feriados = pd.read_excel(pasta_input / 'dados_calendario.xlsx', engine='calamine', sheet_name='Feriados')

# Garantir que as datas estejam no mesmo formato
df_feriados['FERIADO'] = pd.to_datetime(df_feriados['FERIADO'], format="%d/%m/%Y")

# Realizar o merge
df_calendario_ocup = df_calendario_ocup.merge(
    df_feriados[['FERIADO', 'UNIDADE', 'TIPO', 'NORMAL', 'REVEZAMENTO']],
    left_on=['DATA_CALEND', 'UNIDADE'],
    right_on=['FERIADO', 'UNIDADE'],
    how='left'
)

data_agora = datetime.now()
# Inserindo coluna data e hora agora
df_calendario_ocup['DATA_AGORA'] = data_agora

# NOVO_NORMAL: Se TIPO for NaN, retorna OCUPACAO; senão, retorna TIPO NORMAL
df_calendario_ocup['NOVO_NORMAL'] = df_calendario_ocup.apply(lambda row: row['OCUPACAO'] if pd.isna(row['TIPO']) else row['NORMAL'],axis=1)

# NOVO_REVEZAMENTO: Se TIPO for NaN, retorna 1; senão, retorna TIPO REVEZAMENTO
df_calendario_ocup['NOVO_REVEZAMENTO'] = df_calendario_ocup.apply(lambda row: 1 if pd.isna(row['TIPO']) else row['REVEZAMENTO'],axis=1)

# Coluna DIAS_NOR_CALEND: se DATA_CALEND <= DATA_AGORA retorna 0, senão NOVO_NORMAL
df_calendario_ocup['DIAS_NOR_CALEND_'] = df_calendario_ocup.apply(lambda row: 0 if row['DATA_CALEND'] <= row['DATA_AGORA'] else row['NOVO_NORMAL'],axis=1)

# Coluna DIAS_REV_CALEND: se DATA_CALEND <= DATA_AGORA retorna 0, senão NOVO_REVEZAMENTO
df_calendario_ocup['DIAS_REV_CALEND_'] = df_calendario_ocup.apply(lambda row: 0 if row['DATA_CALEND'] <= row['DATA_AGORA'] else row['NOVO_REVEZAMENTO'],axis=1)

# Coluna PARCIAL_HOJE: se DATA_CALEND = DATA_AGORA, RETORNA 1 - (Hora Agora / 24)
hora_agora = data_agora.hour + data_agora.minute / 60 + data_agora.second / 3600
data_hoje = data_agora.date()

df_calendario_ocup['PARCIAL_HOJE'] = df_calendario_ocup['DATA_CALEND'].dt.date.apply(lambda d: 1 - (hora_agora / 24) if d == data_hoje else 0)

df_calendario_ocup['DIAS_NOR_CALEND'] = df_calendario_ocup['DIAS_NOR_CALEND_'] + df_calendario_ocup['PARCIAL_HOJE']
df_calendario_ocup['DIAS_REV_CALEND'] = df_calendario_ocup['DIAS_REV_CALEND_'] + df_calendario_ocup['PARCIAL_HOJE']

bd_calend = df_calendario_ocup.groupby(['MES_CALEND', 'UNIDADE'], as_index=False)[['DIAS_NOR_CALEND', 'DIAS_REV_CALEND']].sum()
bd_calend = bd_calend.sort_values(by=['MES_CALEND', 'UNIDADE'])
bd_calend = bd_calend.rename(columns={
    'DIAS_NOR_CALEND': 'TOT_DIAS_NOR_CALEND',
    'DIAS_REV_CALEND': 'TOT_DIAS_REV_CALEND'
})

bd_calend['TOT_HORAS_NOR_CALEND'] = bd_calend['TOT_DIAS_NOR_CALEND'] * 24
bd_calend['TOT_HORAS_REV_CALEND'] = bd_calend['TOT_DIAS_REV_CALEND'] * 24

print("✅ Geração de tabelas de calendário concluído!")

✅ Geração de tabelas de calendário concluído!


In [11]:
# Inicio Step = GERAR DADOS LTP
bd_prod['NEC_BASE_PROD_PCS'] = (
    bd_prod['estoq_inicial_pcs'] 
    - bd_prod['carteira_mes_atual'] 
    - bd_prod['carteira_arraste_mes_anterior'] 
    - bd_prod['previsao_pcs'] 
    - bd_prod['estoq_seg_pcs']
)

bd_prod['PRIORIDADE'] = 1

bd_mat_param_prioridade_1 = bd_mat_param[bd_mat_param['prioridade'] == 1]

# Join completo entre bd_prod e bd_mat_param
bd_prod_nec = pd.merge(
    bd_mat_param_prioridade_1,
    bd_prod,
    left_on=['produto', 'unidade_fat', 'prioridade'],
    right_on=['cod_produto', 'empresa', 'PRIORIDADE'],
    how='outer'  # FULL OUTER JOIN
)

# Importar aba matriz_regioes da planilha matriz regioes
bd_matriz_regioes = pd.read_excel(pasta_input / 'matriz_regioes.xlsx', sheet_name='matriz_regioes', engine='calamine')

# Merge para adicionar a Região  de Faturamento na base de produtos
bd_prod_nec = bd_prod_nec.merge(
    bd_matriz_regioes,
    how='left',
    left_on='empresa',
    right_on='Unidade'
)

bd_prod_nec = bd_prod_nec.rename(columns={'Unidade': 'Unidade_Fat'})
bd_prod_nec = bd_prod_nec.rename(columns={'Região': 'Reg_Unid_Fat'})

# Merge para adicionar a Região Unidade Produtiva na base de produtos
bd_prod_nec = bd_prod_nec.merge(
    bd_matriz_regioes,
    how='left',
    left_on='unidade_prod',
    right_on='Unidade'
)

bd_prod_nec = bd_prod_nec.rename(columns={'Unidade': 'Unidade_Prod'})
bd_prod_nec = bd_prod_nec.rename(columns={'Região': 'Reg_Unid_Prod'})

# Criando a coluna MESMA REGIAO, com os critérios aplicados
bd_prod_nec['MESMA_REG'] = bd_prod_nec.apply(
    lambda row: "NAO" if pd.isna(row['Reg_Unid_Prod']) else ("SIM" if row['Reg_Unid_Fat'] == row['Reg_Unid_Prod'] else "NAO"),
    axis=1
)

# Inserindo a coluna Mes_Ref_Ant na bd_prod, e criando uma bd_prod_mes_anterior
bd_prod_mes_anterior = bd_prod[['mes_ref', 'empresa', 'cod_produto', 'descricao', 'saldo_previsao_pcs']].copy()
bd_prod_mes_anterior = bd_prod_mes_anterior.rename(columns=
    {'mes_ref': 'MES_REF', 
     'empresa': 'EMPRESA', 
     'cod_produto': 'COD_PROD', 
     'descricao': 'DESC_PROD',
     'saldo_previsao_pcs': 'SALDO_PREV_PROX_MES_PCS'
     }
)

# Criando coluna MES_REF_ANT para definir mes do saldo da previsão
bd_prod_mes_anterior['MES_REF_ANT'] = (bd_prod_mes_anterior['MES_REF'] - pd.DateOffset(months=1)).dt.to_period('M').dt.to_timestamp()

# Adicionar a coluna SALDO_PREV_PROX_MES_PCS na bd_prod_nec
# Criando um dicionário com as chaves de busca e a coluna que você quer trazer
map_dict = bd_prod_mes_anterior.set_index(['MES_REF_ANT', 'EMPRESA', 'COD_PROD'])['SALDO_PREV_PROX_MES_PCS'].to_dict()

# Usando map para trazer a coluna 'SALDO_PREV_PROX_MES_PCS' para bd_prod_nec
bd_prod_nec['SALDO_PREV_PROX_MES_PCS'] = bd_prod_nec[['mes_ref', 'empresa', 'cod_produto']].apply(
    lambda row: map_dict.get((row['mes_ref'], row['empresa'], row['cod_produto']), None), axis=1
)

# Eliminando linhas filtrando mes_ref que não seja null
bd_prod_nec = bd_prod_nec[bd_prod_nec['mes_ref'].notna()]

# Criando a coluna PCS_NEC_PROD_MESMA_REG_SIM_NAO
# Preencher todos os NaN/None das colunas usadas no cálculo com zero
colunas_nec = [
    'carteira_arraste_mes_anterior', 'carteira_mes_atual', 'saldo_previsao_pcs',
    'SALDO_PREV_PROX_MES_PCS', 'estoq_seg_pcs', 'estoq_inicial_pcs', 'estoq_transf_pcs'
]
for col in colunas_nec:
    bd_prod_nec[col] = pd.to_numeric(bd_prod_nec[col], errors='coerce').fillna(0.0)

# Agora pode aplicar o cálculo normalmente
bd_prod_nec['PCS_NEC_PROD_MESMA_REG_SIM_NAO'] = bd_prod_nec.apply(
    lambda row: (
        row['carteira_arraste_mes_anterior'] + row['carteira_mes_atual'] + row['saldo_previsao_pcs'] + row['SALDO_PREV_PROX_MES_PCS'] + row['estoq_seg_pcs']
        - (row['estoq_inicial_pcs'] + row['estoq_transf_pcs'])
    ) if row['tipo_produto'] == 'MR'
    else (
        row['carteira_arraste_mes_anterior'] + row['carteira_mes_atual'] + row['saldo_previsao_pcs'] + row['estoq_seg_pcs']
        - (row['estoq_inicial_pcs'] + row['estoq_transf_pcs'])
    ),
    axis=1
)

# Aplicar substituir null por Zero na coluna SALDO_PREV_PROX_MES_PCS
bd_prod_nec['SALDO_PREV_PROX_MES_PCS'] = bd_prod_nec['SALDO_PREV_PROX_MES_PCS'].fillna(0)

# Padronizando nomes de colunas para concluir bd_prod_nec
bd_prod_nec = bd_prod_nec.rename(columns=
    {'mes_ref': 'MES_REF', 
     'empresa': 'EMPRESA', 
     'cod_produto': 'COD_PROD', 
     'descricao': 'DESC_PROD',
     'linha_prod': 'LINHA_PROD',
     'familia_prod': 'FAMILIA_PROD',
     'tipo_produto': 'TIPO_PROD',
     'curva_abc': 'CURVA_ABC',
     'curva_123': 'CURVA_123',
     'estoq_seg_pcs': 'EST_SEG_PCS',
     'estoq_inicial_pcs': 'EST_INI_PCS',
     'carteira_arraste_mes_anterior': 'CART_ARR_MES_ANT',
     'carteira_mes_atual': 'CART_MES_ATUAL',
     'previsao_pcs': 'PREV_PCS',
     'saldo_previsao_pcs': 'SALDO_PREV_PCS',
     'peso_produto_kg': 'PESO_PROD_KG',
     'estoq_transf_pcs': 'EST_TRANS_PCS',
     'unidade_prod': 'UNID_PROD',
     'MESMA_REG': 'MESMA_REG',
     'saldo_previsao_pcs': 'SALDO_PREV_PCS',
     'lote_econ': 'LOTE_ECON',
     'qtd_emb': 'QTD_EMB',
     'lote_min': 'LOTE_MIN',
     }
)

# Definindo colunas e encerrando processo de formação da bd_prod_nec
bd_prod_nec = bd_prod_nec[
    ['MES_REF', 'EMPRESA', 'UNID_PROD', 'MESMA_REG', 'COD_PROD', 'DESC_PROD', 
     'LINHA_PROD', 'FAMILIA_PROD', 'TIPO_PROD', 'CURVA_ABC','CURVA_123', 'LOTE_ECON','QTD_EMB', 'LOTE_MIN', 
     'EST_SEG_PCS', 'EST_INI_PCS', 'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'PREV_PCS', 
     'SALDO_PREV_PCS', 'PESO_PROD_KG', 'EST_TRANS_PCS','PCS_NEC_PROD_MESMA_REG_SIM_NAO',
     'SALDO_PREV_PROX_MES_PCS']
].reset_index(drop=True)

In [12]:
### Transformação 04_GerarBaseLTP para gerar a base LTP

# Carregar base_dados_roteiros
bd_rot = pd.read_excel(
    pasta_input / 'base_dados_roteiros.xlsx',
    sheet_name='base_dados_roteiros',
    dtype={'cod_produto': str}
)

bd_rot['cod_produto'] = bd_rot['cod_produto'].astype(str).str.strip().str.upper()
bd_rot = bd_rot[bd_rot['cod_produto'].isin(produtos_param)].copy().reset_index(drop=True)

if 'mes_ref' in bd_rot.columns:
    bd_rot['mes_ref'] = pd.to_datetime(bd_rot['mes_ref'], errors='coerce', dayfirst=True)
    
colunas_manter = ['mes_ref', 'empresa', 'cod_produto', 'descricao', 'linha_prod', 'familia_prod', 'tipo_produto', 'cod_ferramenta', 'n_cavidade', 'ciclo_molde', 'pcs_hr', 'mo', 'alocacao_recurso', 'grupo_setor', 'prioridade']

bd_rot = bd_rot[colunas_manter].reset_index(drop=True)

# Adicionando nova coluna na tabela de roteiros
bd_rot['COD_FERR'] = (bd_rot['cod_ferramenta'] + "|" + bd_rot['empresa'].str[:3]).str.upper()

# Faz o merge trazendo somente as 4 colunas desejadas, associando pelos campos de chave
bd_rot = bd_rot.merge(
    bd_calend, 
    how='left', 
    left_on=['mes_ref', 'empresa'], 
    right_on=['MES_CALEND', 'UNIDADE']
)

# Removendo as colunas que não quero na bd_rot
colunas_excluir = ['MES_CALEND', 'UNIDADE']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Carregando o Calendario de Recursos
bd_calend_rec = pd.read_excel(pasta_input / 'calend_recursos.xlsx', sheet_name='calend_recursos')

# Inserindo coluna PER_TURNO no calendário de recursos
bd_calend_rec['PER_TURNO'] = bd_calend_rec['turno_maq'] / 3

# Inserindo colunas do calendário de recursos na bd_rot
bd_rot = bd_rot.merge(
    bd_calend_rec, 
    how='left', 
    left_on=['mes_ref', 'empresa', 'alocacao_recurso'], 
    right_on=['mes_ref', 'unidade', 'maq']
)

colunas_excluir = ['unidade', 'maq', 'setores', 'turno_maq']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

bd_rot.rename(columns={
    'revezamento': 'REC_revezamento',
    'horas_extras': 'REC_horas_extras',
    'disp_mes': 'REC_disp_mes',
    'horas_indisp': 'REC_horas_indisp',
    'PER_TURNO': 'REC_PER_TURNO'
}, inplace=True)

# Carregando Calendário de Ferramentas
bd_calend_ferr = pd.read_excel(pasta_input / 'calend_ferramentas.xlsx', sheet_name='calend_ferramentas')
bd_calend_ferr['NEW_MOLDES'] = bd_calend_ferr['moldes'].str.upper() + "|" + bd_calend_ferr['unidade'].str.upper()

# Merge entre bd_rot e bd_calend_ferr
bd_rot = bd_rot.merge(
    bd_calend_ferr, 
    how='left', 
    left_on=['mes_ref', 'empresa', 'cod_ferramenta'], 
    right_on=['mes_ref', 'unidade', 'moldes']
)

colunas_excluir = ['moldes','unidade', 'setores']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

bd_rot.rename(columns={
    'horas_indisp': 'FER_horas_indisp',
    'disp_mes': 'FER_disp_mes',
    'NEW_MOLDES': 'moldes'
}, inplace=True)

# Preencher apenas os valores nulos de FER_horas_indisp com zero
bd_rot['FER_horas_indisp'] = bd_rot['FER_horas_indisp'].fillna(0)

# Calculando coluna HOR_REC_C1
# Se REC_revezamento for "SIM", calcula com TOT_HORAS_REV_CALEND, senão com TOT_HORAS_NOR_CALEND
# Calculando coluna HOR_REC_C1, se algum valor for nulo, retorna 0
bd_rot['HOR_REC_C1'] = np.where(
    bd_rot[['TOT_HORAS_REV_CALEND', 'REC_horas_extras', 'REC_horas_indisp', 'REC_PER_TURNO', 'REC_disp_mes', 'TOT_HORAS_NOR_CALEND']].isnull().any(axis=1),
    0,
    np.where(
        bd_rot['REC_revezamento'] == "SIM",
        ((bd_rot['TOT_HORAS_REV_CALEND'] + bd_rot['REC_horas_extras'] - bd_rot['REC_horas_indisp']) * bd_rot['REC_PER_TURNO']) * bd_rot['REC_disp_mes'],
        ((bd_rot['TOT_HORAS_NOR_CALEND'] + bd_rot['REC_horas_extras'] - bd_rot['REC_horas_indisp']) * bd_rot['REC_PER_TURNO']) * bd_rot['REC_disp_mes']
    )
).round(2)

# Calculando coluna HOR_FERR_C1
# Se REC_revezamento for "SIM", calcula com TOT_HORAS_REV_CALEND, senão com TOT_HORAS_NOR_CALEND
# Calculando coluna HOR_FER_C1, se algum valor for nulo, retorna 0, senão calcula conforme regra, com duas casas decimais
bd_rot['HOR_FER_C1'] = np.where(
    bd_rot[['TOT_HORAS_REV_CALEND', 'REC_horas_extras', 'FER_horas_indisp', 'TOT_HORAS_NOR_CALEND', 'FER_disp_mes']].isnull().any(axis=1),
    0,
    np.where(
        bd_rot['REC_revezamento'] == "SIM",
        (bd_rot['TOT_HORAS_REV_CALEND'] + (bd_rot['REC_horas_extras'] - bd_rot['FER_horas_indisp'])) * bd_rot['FER_disp_mes'],
        (bd_rot['TOT_HORAS_NOR_CALEND'] + (bd_rot['REC_horas_extras'] - bd_rot['FER_horas_indisp'])) * bd_rot['FER_disp_mes']
    )
).round(2)

# Criando coluna HOR_REC conforme a regra
# Se HOR_REC_C1 for null ou menor que 0, atribui 0; senão, atribui o valor de HOR_REC_C1
bd_rot['HOR_REC'] = np.where(
    bd_rot['HOR_REC_C1'].isnull() | (bd_rot['HOR_REC_C1'] < 0),
    0,
    bd_rot['HOR_REC_C1']
).round(2)

# Criando coluna HOR_FER conforme a regra
# Se HOR_FER_C1 for null ou menor que 0, atribui 0; senão, atribui o valor de HOR_FER_C1
bd_rot['HOR_FER'] = np.where(
    bd_rot['HOR_FER_C1'].isnull() | (bd_rot['HOR_FER_C1'] < 0),
    0,
    bd_rot['HOR_FER_C1']
).round(2)

# Criando coluna HOR_CAP conforme a regra
# Se HOR_REC for menor que HOR_FER, atribui HOR_REC; senão, atribui HOR_FER
bd_rot['HOR_CAP'] = np.where(
    bd_rot['HOR_REC'] < bd_rot['HOR_FER'],
    bd_rot['HOR_REC'],
    bd_rot['HOR_FER']
).round(2)

# Eliminando colunas para tornar mais clean a bd_rot
colunas_excluir = [
    'REC_revezamento', 'REC_horas_extras', 'REC_disp_mes', 'REC_horas_indisp',
    'TOT_HORAS_REV_CALEND', 'TOT_HORAS_NOR_CALEND', 'REC_PER_TURNO',
    'FER_horas_indisp', 'HOR_REC_C1', 'HOR_FER_C1', 'disp_mes'
]
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando a coluna prioridade do bd_rot para PRIORIDADE_ROTEIRO, para evitar confusão 
# com a prioridade da bd_mat_param
bd_rot.rename(columns={'prioridade': 'PRIORIDADE_ROTEIRO'}, inplace=True)

# Executando o merge entre bd_rot e bd_mat_param para trazer os campos unidade_fat e prioridade
bd_rot = bd_rot.merge(
    bd_mat_param, 
    how='left', 
    left_on=['empresa', 'cod_produto'], 
    right_on=['unidade_prod', 'produto']
)

# Elimnando campos que não são mais necessários da bd_rot
colunas_excluir = ['produto', 'unidade_prod', 'tipo_abast']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando colunas da matriz parametros que foram trazidas para a bd_rot
bd_rot.rename(columns={
    'unidade_fat': 'MAT_PAR_unidade_fat',
    'prioridade': 'MAT_PAR_prioridade'
}, inplace=True)


# Executando o merge entre bd_rot e bd_prod_necs para trazer os campos de volumese e dados
bd_rot = bd_rot.merge(
    bd_prod_nec, 
    how='left', 
    left_on=['mes_ref', 'MAT_PAR_unidade_fat', 'cod_produto'], 
    right_on=['MES_REF', 'EMPRESA', 'COD_PROD']
)

# Elimnando campos que não são mais necessários da bd_rot
colunas_excluir = [
    'TOT_DIAS_NOR_CALEND','TOT_DIAS_REV_CALEND', 'moldes', 'MES_REF', 'EMPRESA',
    'UNID_PROD', 'COD_PROD', 'DESC_PROD', 'LINHA_PROD',
    'FAMILIA_PROD', 'TIPO_PROD', 'CURVA_ABC', 'CURVA_123'
]
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando da bd_roteiro que foram trazidas para a bd_rot
bd_rot.rename(columns={
    'mes_ref': 'MES_REF',
    'empresa': 'EMPRESA',
    'cod_produto': 'COD_PROD',
    'descricao': 'DESC_PROD',
    'linha_prod': 'LINHA_PROD',
    'familia_prod': 'FAMILIA_PROD',
    'tipo_produto': 'TIPO_PROD',
    'COD_FERR': 'COD_FER_UNID',
    'cod_ferramenta': 'COD_FERR',
    'n_cavidade': 'N_CAVIDADE',
    'ciclo_molde': 'CICLO_MOLDE',
    'pcs_hr': 'PCS_HORA',
    'mo': 'MO',
    'alocacao_recurso': 'ALOC_REC',
    'grupo_setor': 'GRUPO_SETOR',
    'peso_produto_kg': 'PESO_PROD',
    'PRIORIDADE_ROTEIRO': 'PRIOR_ROT',
    'MAT_PAR_unidade_fat': 'UNID_FAT_MATPAR',
    'MAT_PAR_prioridade': 'PRIOR_MATPAR'
}, inplace=True)

# Executando o merge entre bd_rot e bd_matriz_regioes para trazer os campo região fat
bd_rot = bd_rot.merge(
    bd_matriz_regioes, 
    how='left', 
    left_on=['UNID_FAT_MATPAR'], 
    right_on=['Unidade']
)

# Elimnando campos que não são mais necessários da bd_rot
bd_rot = drop_colunas(bd_rot, ['Unidade'])

# Renomeando da campos que foram trazidos para a bd_rot
bd_rot.rename(columns={'Região': 'REG_UNID_FAT'}, inplace=True)

# Executando o merge entre bd_rot e bd_matriz_regioes para trazer os campo região prod
bd_rot = bd_rot.merge(
    bd_matriz_regioes, 
    how='left', 
    left_on=['EMPRESA'], 
    right_on=['Unidade']
)

# Elimnando campos que não são mais necessários da bd_rot
bd_rot = drop_colunas(bd_rot, ['Unidade'])

# Renomeando da campos que foram trazidos para a bd_rot
bd_rot.rename(columns={'Região': 'REG_UNID_PROD'}, inplace=True)

# Criando o campo MESMA_REG conforme a regra solicitada
bd_rot['MESMA_REG'] = np.where(
    bd_rot['REG_UNID_PROD'].isna(),
    "NAO",
    np.where(
        bd_rot['REG_UNID_FAT'] == bd_rot['REG_UNID_PROD'],
        "SIM",
        "NAO"
    )
)

# Substituir null (NaN) por 0 nos campos solicitados
campos_zerar = [
    'EST_SEG_PCS', 'EST_INI_PCS',
    'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'PREV_PCS', 'SALDO_PREV_PCS',
    'PESO_PROD_KG', 'EST_TRANS_PCS', 'SALDO_PREV_PROX_MES_PCS', 'PCS_NEC_PROD_MESMA_REG_SIM_NAO'
]
bd_rot[campos_zerar] = bd_rot[campos_zerar].fillna(0)

# Criando a coluna MES_REF_ANT reduzindo 1 mês de MES_REF
bd_rot['MES_REF_ANT'] = (bd_rot['MES_REF'] - pd.DateOffset(months=1)).dt.to_period('M').dt.to_timestamp()

# Renomeando da bd_roteiro que foram trazidas para a bd_rot
bd_rot.rename(columns={
    'EMPRESA': 'UNID_PROD',
    'UNID_FAT_MATPAR': 'UNID_FAT',
}, inplace=True)

# Importar aba matriz_regioes da planilha matriz limitantes
bd_matriz_limitantes = pd.read_excel(pasta_input / 'matriz_limitantes.xlsx', sheet_name='matriz_limitantes', dtype={'produto': str})

if 'mes_ref' in bd_matriz_limitantes.columns:
    bd_matriz_limitantes['mes_ref'] = pd.to_datetime(bd_matriz_limitantes['mes_ref'], errors='coerce', dayfirst=True)

# Executando o merge entre bd_rot e bd_matriz_limitantes para trazer os campo limitante_pcs
bd_rot = bd_rot.merge(
    bd_matriz_limitantes, 
    how='left',
    left_on=['MES_REF','UNID_FAT','COD_PROD'],
    right_on=['mes_ref','unidade','produto']
)

# Elimnando campos que não são mais necessários da bd_rot
colunas_excluir = ['mes_ref','unidade','produto']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando da campos que foram trazidos para a bd_rot
bd_rot.rename(columns={'limitante_pcs': 'LIMIT_PCS'}, inplace=True)

# Aplicando a regra no campo LIMIT_PCS conforme lógica Java
bd_rot['AVALIAR'] = bd_rot['PRIOR_MATPAR'].astype(str) + "|" + bd_rot['PRIOR_ROT'].astype(str)
bd_rot['LIMIT_PCS'] = np.where(
    (bd_rot['AVALIAR'] != "1|1") | (bd_rot['LIMIT_PCS'].isnull()),
    0,
    bd_rot['LIMIT_PCS']
)

# Formatando a coluna PRIOR_MATPAR para inteiro
bd_rot['PRIOR_MATPAR'] = bd_rot['PRIOR_MATPAR'].fillna(0).astype(int)

# Eliminar a coluna AVALIAR
colunas_excluir = ['AVALIAR', 'MES_REF_ANT']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Criar ID_COMP
bd_rot['ID_COMP'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_PROD'].str.upper() + "|" +
    bd_rot['COD_PROD'].str.upper()
)

# Criar ID_RECURSO
bd_rot['ID_RECURSO'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_PROD'].str.upper() + "|" +
    bd_rot['ALOC_REC'].str.upper()
)

# Criar ID_FERRAMENTA
bd_rot['ID_FERRAMENTA'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_PROD'].str.upper() + "|" +
    bd_rot['COD_FER_UNID'].str.upper()
)

# Criar ID_PROD_UNID_FAT
bd_rot['ID_PROD_UNID_FAT'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_FAT'].str.upper() + "|" +
    bd_rot['COD_PROD'].astype(str).str.upper()
)

# Criar ID_PROD_UNID_FAT_ANT já calculando o mês anterior no mesmo comando
bd_rot['ID_PROD_UNID_FAT_ANT'] = (
    (bd_rot['MES_REF'] - pd.DateOffset(months=1)).dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_FAT'].str.upper()  + "|" +
    bd_rot['COD_PROD'].str.upper()
)
 
# Criar a coluna IND
# Classificação antiga
# bd_rot = bd_rot.sort_values(by=['MES_REF', 'UNID_FAT', 'COD_PROD', 'PRIOR_MATPAR', 'PRIOR_ROT', 'TIPO_PROD']).reset_index(drop=True)

# Classificação nova
bd_rot = bd_rot.sort_values(by=['MES_REF', 'TIPO_PROD', 'UNID_FAT', 'COD_PROD', 'PRIOR_MATPAR', 'PRIOR_ROT']).reset_index(drop=True)
bd_rot['IND'] = range(1, len(bd_rot) + 1)

# |||||||||||||||||||||||||||||||| ID_NUM_REC ||||||||||||||||||||||||||||||||
# Remover linhas onde ID_RECURSO é NaN
bd_rot = bd_rot[bd_rot['ID_RECURSO'].notna()].copy()

# Ordena primeiro por ID_RECURSO e depois por IND
# bd_rot = bd_rot.sort_values(by=['ID_RECURSO', 'IND']).reset_index(drop=True)

# Cria o índice incremental por ID_RECURSO, começando em 1
# bd_rot['ID_NUM_REC'] = (bd_rot.groupby('ID_RECURSO').cumcount() + 1).astype(int)

# Classificar crescente pela coluna IND
bd_rot = bd_rot.sort_values(by=['IND']).reset_index(drop=True)
# |||||||||||||||||||||||||||||||| ID_NUM_REC ||||||||||||||||||||||||||||||||

# |||||||||||||||||||||||||||||||| ID_NUM_FER ||||||||||||||||||||||||||||||||
# Remover linhas onde ID_FERRAMENTA é NaN
bd_rot = bd_rot[bd_rot['ID_FERRAMENTA'].notna()].copy()

# # Ordena primeiro por ID_NUM_FER e depois por IND
# bd_rot = bd_rot.sort_values(by=['ID_FERRAMENTA', 'IND']).reset_index(drop=True)

# # Cria o índice incremental por ID_NUM_FER, começando em 1
# bd_rot['ID_NUM_FER'] = (bd_rot.groupby('ID_FERRAMENTA').cumcount() + 1).astype(int)

# Classificar crescente pela coluna IND
bd_rot = bd_rot.sort_values(by=['IND']).reset_index(drop=True)
# |||||||||||||||||||||||||||||||| ID_NUM_FER ||||||||||||||||||||||||||||||||

# Criar coluna ID_CONC_REC = ID_RECURSO + "|" + ID_NUM_REC e ID_CONC_FER = ID_FERRAMENTA + "|" + ID_NUM_FER
# bd_rot['ID_CONC_REC'] = bd_rot['ID_RECURSO'] + "|" + bd_rot['ID_NUM_REC'].astype(str)
# bd_rot['ID_CONC_FER'] = bd_rot['ID_FERRAMENTA'] + "|" + bd_rot['ID_NUM_FER'].astype(str)

# Organizando Layout Final da para dados para bd_base_LTP
# Ordenando colunas
nova_ordem_colunas = [
    'MES_REF', 'UNID_FAT', 'UNID_PROD', 'MESMA_REG', 'PRIOR_MATPAR', 'PRIOR_ROT', 'COD_PROD', 'DESC_PROD', 'LINHA_PROD',
    'FAMILIA_PROD', 'TIPO_PROD', 'COD_FER_UNID', 'N_CAVIDADE', 'CICLO_MOLDE', 'PCS_HORA', 'MO', 'ALOC_REC', 
    'GRUPO_SETOR', 'PESO_PROD_KG', 'LOTE_ECON', 'QTD_EMB','LOTE_MIN', 'IND', 'ID_COMP', 'ID_RECURSO', 'ID_FERRAMENTA', 'ID_PROD_UNID_FAT', 'ID_PROD_UNID_FAT_ANT', 'HOR_REC', 'HOR_FER', 'HOR_CAP', 'PREV_PCS', 'EST_SEG_PCS', 'EST_INI_PCS', 'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'SALDO_PREV_PCS', 'EST_TRANS_PCS', 'SALDO_PREV_PROX_MES_PCS', 'LIMIT_PCS'
    ]

# # Reordenando as colunas
bd_prod_rot = bd_rot[nova_ordem_colunas].reset_index(drop=True)

# Criar tabela com ID_PROD_UNID_FAT e MESMA_REG, filtrando PRIOR_MATPAR = 1 e PRIOR_ROT = 1
bd_prod_rot_PRIOR_11 = bd_prod_rot[
    (bd_prod_rot['PRIOR_MATPAR'] == 1) & (bd_prod_rot['PRIOR_ROT'] == 1)
][['ID_PROD_UNID_FAT', 'MESMA_REG']].drop_duplicates().reset_index(drop=True)
bd_prod_rot_PRIOR_11.rename(columns={'MESMA_REG': 'MESMA_REG_PRIOR_11'}, inplace=True)

# Fazer join entre bd_prod_rot e bd_prod_rot_PRIOR_11 para trazer MESMA_REG_PRIOR_11
bd_prod_rot = bd_prod_rot.merge(
    bd_prod_rot_PRIOR_11,
    how='left',
    on='ID_PROD_UNID_FAT'
)

# Substituir coluna MESMA_REG por MESMA_REG_PRIOR_11
bd_prod_rot['MESMA_REG'] = bd_prod_rot['MESMA_REG_PRIOR_11'].fillna(bd_prod_rot['MESMA_REG'])
# Eliminar coluna MESMA_REG_PRIOR_11
bd_prod_rot = drop_colunas(bd_prod_rot, ['MESMA_REG_PRIOR_11'])

# Criar bd_estrutura_filtrada, com base na bd_estruturas eliminando cod_insumo que não constam na coluna COD_PROD da bd_prod_rot
# codigos_validos = set(bd_prod_rot['COD_PROD'].unique())
# bd_estrutura_filtrada = bd_estruturas[bd_estruturas['cod_insumo'].isin(codigos_validos)].copy().reset_index(drop=True)

# Criar conjunto de tuplas válidas (COD_PROD, UNID_PROD)
codigos_validos = set(zip(bd_prod_rot['COD_PROD'], bd_prod_rot['UNID_PROD']))

# Filtrar estrutura com base em cod_insumo + empresa
bd_estrutura_filtrada = (
    bd_estruturas[
        bd_estruturas[['cod_insumo', 'empresa']].apply(tuple, axis=1).isin(codigos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

# Mesclar bd_base_LTP e bd_estrutura_filtrada por UNID_PROD, COD_PROD vs empresa e cod_prod_acabado
bd_prod_rot_estr = pd.merge(bd_prod_rot, bd_estrutura_filtrada, how='left', left_on=['UNID_PROD', 'COD_PROD'], right_on=['empresa', 'cod_prod_acabado']).reset_index(drop=True)

# Função para criar estrutura com fator estrutural
bd_estr_fator_estrutural = criar_estrutura_com_fator_estrutural(bd_estrutura_filtrada)

print("✅ Geração de tabelas de dados LTP concluída!")

✅ Geração de tabelas de dados LTP concluída!


In [13]:
########### TRAZER CAMPOS DA MATRIZ TRANSFERENCIA E ESTOQUE ORIGEM ###########
bd_LTP_NEC = bd_prod_rot.copy()

# Trazer para bd_produtos_nec as coluna de status da tabela bd_mat_transf_comp, colunas considera_estoq_origem e considera_estoq_triangulacao

 # Carregar matriz_transf_componentes
bd_mat_transf_comp = pd.read_excel(
    pasta_input / 'matriz_transf_componentes.xlsx',
    sheet_name='matriz_transf_componentes',
    dtype={'cod_produto': str}
)
bd_mat_transf_comp['cod_produto'] = bd_mat_transf_comp['cod_produto'].astype(str).str.strip().str.upper()
# Filtrar o DataFrame bd_mat_transf_comp para manter apenas as linhas onde 'validacao' == 'Sim'
bd_mat_transf_comp = bd_mat_transf_comp[bd_mat_transf_comp['validacao'].str.upper() == 'SIM']
bd_mat_transf_comp = bd_mat_transf_comp.reset_index(drop=True)

bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_mat_transf_comp[['unid_destino', 'unid_origem', 'unid_triangulacao', 'cod_produto', 'considera_estoq_origem', 'considera_estoq_triangulacao']],
    how='left',
    left_on=['UNID_FAT', 'COD_PROD'],
    right_on=['unid_destino', 'cod_produto']
)

# Trazer os valores de Estoque Inicial e Estoque Transferencia das unidades Origem
# Cria uma cópia da tabela para servir como origem
bd_produtos_origem = bd_LTP_NEC[['UNID_FAT', 'MES_REF', 'COD_PROD', 'EST_INI_PCS', 'EST_TRANS_PCS']].copy()

# Remover duplicatas bd_produtos_origem
bd_produtos_origem = bd_produtos_origem.drop_duplicates(subset=['UNID_FAT', 'MES_REF', 'COD_PROD']).reset_index(drop=True)

bd_produtos_origem = bd_produtos_origem.rename(columns={
    'UNID_FAT': 'UNID_FAT_ORIGEM',
    'MES_REF': 'MES_REF_ORIGEM',
    'COD_PROD': 'COD_PROD_ORIGEM',
    'EST_INI_PCS': 'EST_INI_PCS_ORIGEM',
    'EST_TRANS_PCS': 'EST_TRANS_PCS_ORIGEM'
})

# Eliminar duplicatas da tabela de origem
bd_produtos_origem = bd_produtos_origem.drop_duplicates(subset=['UNID_FAT_ORIGEM', 'MES_REF_ORIGEM', 'COD_PROD_ORIGEM', 'EST_INI_PCS_ORIGEM', 'EST_TRANS_PCS_ORIGEM'])

# Faz o merge na própria tabela, buscando os valores da origem
bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_produtos_origem,
    how='left',
    left_on=['unid_origem', 'MES_REF', 'COD_PROD'],
    right_on=['UNID_FAT_ORIGEM', 'MES_REF_ORIGEM', 'COD_PROD_ORIGEM']
)

# Criar uma tabela com cópia na bd_produtos_nec ter somente as colunas de interesse e buscar os valores de estoque origem e estoque triangulação
bd_produtos_estoque_origem_triangulacao = bd_LTP_NEC
colunas_excluir = ['considera_estoq_origem', 'considera_estoq_triangulacao']
bd_produtos_estoque_origem_triangulacao = drop_colunas(bd_produtos_estoque_origem_triangulacao, colunas_excluir)


# Trazer para bd_produtos_nec as colunas de status da tabela bd_mat_transf_comp, colunas considera_estoq_origem e considera_estoq_triangulacao
bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao.merge(
    bd_mat_transf_comp[['unid_origem', 'cod_produto', 'considera_estoq_origem']],
    how='left',
    left_on=['UNID_FAT', 'COD_PROD'],
    right_on=['unid_origem', 'cod_produto']
)

# Excluir as colunas unid_triangulacao e cod_produto
colunas_excluir = ['unid_triangulacao', 'cod_produto']
bd_produtos_estoque_origem_triangulacao = drop_colunas(bd_produtos_estoque_origem_triangulacao, colunas_excluir)

bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao.merge(
    bd_mat_transf_comp[['unid_triangulacao', 'cod_produto', 'considera_estoq_triangulacao']],
    how='left',
    left_on=['UNID_FAT', 'COD_PROD'],
    right_on=['unid_triangulacao', 'cod_produto']
)

# Excluir as colunas unid_destino e cod_produto
colunas_excluir = ['unid_triangulacao', 'cod_produto']
bd_produtos_estoque_origem_triangulacao = drop_colunas(bd_produtos_estoque_origem_triangulacao, colunas_excluir)

bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao[['MES_REF', 'UNID_FAT', 'COD_PROD', 'EST_INI_PCS', 'EST_TRANS_PCS','considera_estoq_origem', 'considera_estoq_triangulacao']]

# Eliminar duplicatas da bd_produtos_estoque_origem_triangulacao
bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao.drop_duplicates(subset=['MES_REF', 'UNID_FAT', 'COD_PROD']).reset_index(drop=True, inplace=True)

# Trazer os valores de Estoque Inicial e Estoque Transferencia das unidades Triangulação
# Cria uma cópia da tabela para servir como Triangulação
bd_produtos_triangulacao = bd_LTP_NEC[['UNID_FAT', 'MES_REF', 'COD_PROD', 'EST_INI_PCS', 'EST_TRANS_PCS']].copy()

# Renomeando e organizando colunas bd_produtos_triangulacao
bd_produtos_triangulacao = bd_produtos_triangulacao.rename(columns={
    'UNID_FAT': 'UNID_FAT_TRIANG',
    'MES_REF': 'MES_REF_TRIANG',
    'COD_PROD': 'COD_PROD_TRIANG',
    'EST_INI_PCS': 'EST_INI_PCS_TRIANG',
    'EST_TRANS_PCS': 'EST_TRANS_PCS_TRIANG'
})

# Remover Duplicatas da tabela de triangulação
bd_produtos_triangulacao = bd_produtos_triangulacao.drop_duplicates(subset=['UNID_FAT_TRIANG', 'MES_REF_TRIANG', 'COD_PROD_TRIANG']).reset_index(drop=True)

# Faz o merge na própria tabela, buscando os valores da TRIANG
bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_produtos_triangulacao,
    how='left',
    left_on=['unid_triangulacao', 'MES_REF', 'COD_PROD'],
    right_on=['UNID_FAT_TRIANG', 'MES_REF_TRIANG', 'COD_PROD_TRIANG']
)

# Somar valores Estoque Inicial e Transferencia da Origem
def ORI_TOT_PCS(row):
    if str(row.get('considera_estoq_origem', '')).strip().upper() == 'SIM':
        return row.get('EST_INI_PCS_ORIGEM', 0) + row.get('EST_TRANS_PCS_ORIGEM', 0)
    else:
        return 0
bd_LTP_NEC['ORI_TOT_PCS'] = bd_LTP_NEC.apply(ORI_TOT_PCS, axis=1)

# Somar valores Estoque Inicial e Transferencia da Triangulação
def TRIANG_TOT_PCS(row):
    if str(row.get('considera_estoq_triangulacao', '')).strip().upper() == 'SIM':
        return row.get('EST_INI_PCS_TRIANG', 0) + row.get('EST_TRANS_PCS_TRIANG', 0)
    else:
        return 0
bd_LTP_NEC['TRIANG_TOT_PCS'] = bd_LTP_NEC.apply(TRIANG_TOT_PCS, axis=1)

# Eliminar colunas desnecessárias da bd_LTP_NEC
colunas_excluir = [
    'unid_destino', 'unid_origem', 'unid_triangulacao', 'cod_produto',
    'considera_estoq_origem', 'considera_estoq_triangulacao',
    'UNID_FAT_ORIGEM', 'MES_REF_ORIGEM', 'COD_PROD_ORIGEM',
    'EST_INI_PCS_ORIGEM', 'EST_TRANS_PCS_ORIGEM',
    'UNID_FAT_TRIANG', 'MES_REF_TRIANG', 'COD_PROD_TRIANG',
    'EST_INI_PCS_TRIANG', 'EST_TRANS_PCS_TRIANG'
]
bd_LTP_NEC = drop_colunas(bd_LTP_NEC, colunas_excluir)

# Aplicar zero nos campos vazios de NEC_PCS, ORI_TOT_PCS e TRIANG_TOT_PCS
colunas_preencher = ['ORI_TOT_PCS', 'TRIANG_TOT_PCS']
bd_LTP_NEC[colunas_preencher] = bd_LTP_NEC[colunas_preencher].fillna(0).astype(float)

# Filtrar bd_LTP_NEC pelo campo MAT_PAR_prioridade > 0
bd_LTP_NEC = bd_LTP_NEC[bd_LTP_NEC['PRIOR_MATPAR'] > 0].reset_index(drop=True)

#*****************************************# ID_ULT_PRIORI #*****************************************
# Criar a tabela bd_Ultimo_Roteiro_MAT_PAR com os campos ID_PROD_UNID_FAT, PRIOR_MATPAR e PRIOR_ROT, eliminando as duplicatas
bd_Ultimo_Roteiro_MAT_PAR = bd_LTP_NEC[['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT']]
bd_Ultimo_Roteiro_MAT_PAR = bd_Ultimo_Roteiro_MAT_PAR.sort_values(
    by=['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT'],
    ascending=[True, True, False]
).reset_index(drop=True)
bd_Ultimo_Roteiro_MAT_PAR = bd_Ultimo_Roteiro_MAT_PAR.drop_duplicates(subset='ID_PROD_UNID_FAT', keep='first').reset_index(drop=True)

# Criar a coluna ID_ULT_PRIORI fazendo o merge entre bd_LTP_NEC e bd_Ultimo_Roteiro_MAT_PAR para trazer dados a coluna ID_PROD_UNID_FAT
bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_Ultimo_Roteiro_MAT_PAR[['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT']].rename(columns={'ID_PROD_UNID_FAT': 'ID_ULT_PRIORI'}),
    how='left',
    left_on=['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT'],
    right_on=['ID_ULT_PRIORI', 'PRIOR_MATPAR', 'PRIOR_ROT']
)

bd_LTP_NEC = bd_LTP_NEC.sort_values(['IND']).reset_index(drop=True)

# Zerar valores campo SALDO_PREV_PROX_MES_PCS, considerando se MESMA_REG for igual a "SIM", manter valor, se MESMA_REG for "NAO", zerar o valor
bd_LTP_NEC['SALDO_PREV_PROX_MES_PCS'] = np.where(
    bd_LTP_NEC['MESMA_REG'] == 'NAO',
    bd_LTP_NEC['SALDO_PREV_PROX_MES_PCS'],
    0
)

# Criando cópia dos campos e adicinar nos rótulos, no incio do nome LTP
campos_copiar = [
    'EST_SEG_PCS', 'EST_INI_PCS', 'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'SALDO_PREV_PCS', 'EST_TRANS_PCS','SALDO_PREV_PROX_MES_PCS'
]

for col in campos_copiar:
    bd_LTP_NEC['LTP_' + col] = bd_LTP_NEC[col]
    
# Adicionar coluna LTP_COMP_NEC_PCS com valor 0
bd_LTP_NEC['LTP_COMP_NEC_PCS'] = 0

# --- Utilizando Flags Parametros Lote Minimo e Multiplo Embalagens e Calculando os campos VAR_NEC1, VAR_NEC2 e VAR_NEC3 ---
def carregar_flags_ltp(pasta_painel):
    def carregar_planilha_segura(caminho: Path, **kwargs):
        try:
            return load_workbook(caminho, **kwargs)
        except PermissionError:
            with tempfile.NamedTemporaryFile(suffix=caminho.suffix, delete=False) as tmp:
                shutil.copy2(caminho, tmp.name)
                return load_workbook(tmp.name, **kwargs)

    def obter_valor_nome_definido(wb, nome_definido):
        nome_planilha, intervalo = next(nome_definido.destinations)
        planilha = wb[nome_planilha]
        coluna_min, linha_min, _, _ = range_boundaries(intervalo)
        return planilha.cell(linha_min, coluna_min).value

    arquivo_painel = pasta_painel / 'Painel_LTP.xlsm'
    planilha_ltp = carregar_planilha_segura(arquivo_painel, data_only=True)
    nome_lote_min = planilha_ltp.defined_names['FlagLoteMinimo']
    nome_multiplo_emb = planilha_ltp.defined_names['FlagMultiploEmb']
    lote_min = obter_valor_nome_definido(planilha_ltp, nome_lote_min)
    multiplo_emb = obter_valor_nome_definido(planilha_ltp, nome_multiplo_emb)
    return str(lote_min).strip().upper(), str(multiplo_emb).strip().upper()

lote_min_flag, multiplo_emb_flag = carregar_flags_ltp(pasta_painel)

# Mover Colunas ID_RECURSO e id_FERRAMENTA para o final do DataFrame, depois da coluna LTP_COMP_NEC_PCS
bd_LTP_NEC = bd_LTP_NEC[
    [col for col in bd_LTP_NEC.columns if col not in ['ID_RECURSO', 'ID_FERRAMENTA']] +
    ['ID_RECURSO', 'ID_FERRAMENTA']
]

# Mover colunas HOR_REC, HOR_FER, HOR_CAP para o final do DataFrame, depois da coluna LTP_COMP_NEC_PCS
bd_LTP_NEC = bd_LTP_NEC[
    [col for col in bd_LTP_NEC.columns if col not in ['HOR_REC', 'HOR_FER']] +
    ['HOR_REC', 'HOR_FER']
]

# Excluir coluna HOR_CAP
bd_LTP_NEC = drop_colunas(bd_LTP_NEC, ['HOR_CAP'])

# criar coluna COMP com valor 'SIM' ou 'NAO' dependendo se o COD_PROD está na tabela bd_estruturas
insumos = set(bd_estruturas['cod_insumo'].astype(str).str.strip().str.upper())
bd_LTP_NEC['COMP'] = (bd_LTP_NEC['COD_PROD'].astype(str).str.strip().str.upper().isin(insumos).map({True: 'SIM', False: 'NAO'}))

print("✅ Primeiro Calculo NEC_PCS e Distribuição Capacidade concluídos!")

✅ Primeiro Calculo NEC_PCS e Distribuição Capacidade concluídos!


In [14]:
VALIDOS = {"RECURSO", "FERRAMENTA"}

def carregar_cortes(pasta_painel: Path):

    def carregar_planilha_segura(caminho: Path, **kwargs):
        try:
            return load_workbook(caminho, **kwargs)
        except PermissionError:
            with tempfile.NamedTemporaryFile(suffix=caminho.suffix, delete=False) as tmp:
                shutil.copy2(caminho, tmp.name)
                return load_workbook(tmp.name, **kwargs)

    def obter_valor_nome_definido(wb, nome_definido):
        nome_planilha, intervalo = next(nome_definido.destinations)
        planilha = wb[nome_planilha]
        coluna_min, linha_min, _, _ = range_boundaries(intervalo)
        return planilha.cell(linha_min, coluna_min).value

    def limpar_valor(x):
        if x is None:
            return ""
        if isinstance(x, float) and math.isnan(x):
            return ""
        return str(x).strip().upper()

    arquivo_painel = pasta_painel / 'Painel_LTP.xlsm'
    planilha_ltp = carregar_planilha_segura(arquivo_painel, data_only=True)

    try:
        nome_primeiro_corte = planilha_ltp.defined_names['rgPrimeiroCorte']
        nome_segundo_corte  = planilha_ltp.defined_names['rgSegundoCorte']
    except KeyError as e:
        raise RuntimeError(
            "Ranges nomeadas 'rgPrimeiroCorte' ou 'rgSegundoCorte' não encontradas no Painel_LTP.xlsm."
        ) from e

    rgPrimeiroCorte = limpar_valor(obter_valor_nome_definido(planilha_ltp, nome_primeiro_corte))
    rgSegundoCorte  = limpar_valor(obter_valor_nome_definido(planilha_ltp, nome_segundo_corte))

    # 🔴 VALIDAÇÃO OBRIGATÓRIA
    if not rgPrimeiroCorte or not rgSegundoCorte:
        raise ValueError(
            "Defina obrigatoriamente os campos 'Primeiro Corte' e 'Segundo Corte' no Painel_LTP.\n"
            "Valores permitidos: RECURSO ou FERRAMENTA."
        )

    if rgPrimeiroCorte not in VALIDOS:
        raise ValueError(
            f"Valor inválido em rgPrimeiroCorte: '{rgPrimeiroCorte}'. "
            "Valores permitidos: RECURSO ou FERRAMENTA."
        )

    if rgSegundoCorte not in VALIDOS:
        raise ValueError(
            f"Valor inválido em rgSegundoCorte: '{rgSegundoCorte}'. "
            "Valores permitidos: RECURSO ou FERRAMENTA."
        )

    if rgPrimeiroCorte == rgSegundoCorte:
        raise ValueError(
            "Primeiro Corte e Segundo Corte não podem ser iguais.\n"
            "Escolha uma sequência válida entre RECURSO e FERRAMENTA."
        )

    return rgPrimeiroCorte, rgSegundoCorte

# chamada
rgPrimeiroCorte, rgSegundoCorte = carregar_cortes(pasta_painel)

In [15]:
# Se IND repetido, parar o código e gerar print de aviso, indicando quais são os IND repetidos para análise
if bd_LTP_NEC['IND'].duplicated().any():
    inds_duplicados = bd_LTP_NEC[bd_LTP_NEC['IND'].duplicated()]['IND'].unique()
    print(f"⚠️ Atenção: IND duplicados encontrados! IND's repetidos: {inds_duplicados}")
    raise ValueError("IND duplicados encontrados. Verifique os dados antes de continuar.")

In [17]:
# ======================================================================================
# CELL 8 - MOTOR DE CORTE SOLVER (M1)
# ======================================================================================
# Drop-in replacement para a maquina iterativa de cortes.
# Mantem a ETL anterior intacta e usa as variaveis ja geradas:
# bd_LTP_NEC, bd_estrutura_filtrada, bd_estr_fator_estrutural,
# lote_min_flag, multiplo_emb_flag, rgPrimeiroCorte e rgSegundoCorte.
#
# Premissa preservada: o pipeline fecha o primeiro mes antes de processar os demais.
#
# ROTEIRO DA CELULA
# 1) Configura parametros de debug e limites do solver.
# 2) Define a ordem dos buckets cortaveis, mantendo a regra do motor antigo.
# 3) Cria helpers pequenos para normalizacao, dimensoes e recalculo do modelo mensal.
# 4) Recalcula o M1 a partir de bd_LTP_NEC, gerando o snapshot antes dos cortes.
# 5) Em cada ciclo macro, resolve RECURSO e FERRAMENTA na ordem definida no painel.
# 6) Para cada gargalo, monta bd_ECO de corte direto e bd_ECO PA -> PI.
# 7) Aplica cortes nos campos LTP de origem, recalcula o modelo e mede residuo.
# 8) Gera tabelas de auditoria e debug para analisar o que foi cortado e o que sobrou.
# ======================================================================================

# -----------------------------
# 8.0 Configuracoes de debug
# -----------------------------
DEBUG_SOLVER = True
DEBUG_RECURSO = None       # Exemplo: "26725"
DEBUG_FERRAMENTA = None    # Exemplo: "0103K|MAT"
DEBUG_PRODUTO = None       # Exemplo: "123456"

SOLVER_MAX_CICLOS_MES = 6
SOLVER_TOL_HR = 1e-6
SOLVER_EXPORTAR_DEBUG = True
SOLVER_EXPORTAR_LOG_DETALHADO_XLSX = True  # Gera log detalhado em Excel para validacao do usuario.

# Sequencia de corte vem do Painel_LTP.xlsm.
SEQUENCIA_CORTES_SOLVER = [rgPrimeiroCorte, rgSegundoCorte]

# Ordem de buckets preservando a matriz logica atual.
# Cada item informa:
# - ORDEM: prioridade relativa dentro do gargalo.
# - MESMA_REG: se o bucket vale para atendimento na mesma regiao ou nao.
# - BUCKET: nome amigavel usado no log.
# - HR_COL/PCS_COL: colunas de decomposicao calculadas antes do corte.
# - LTP_COL: coluna real que sera reduzida quando o corte for aplicado.
# DIF_LM/DIF_EMB ficam fora do corte por padrao: se sobrarem, aparecem como residuo nao cortavel.
BUCKETS_CORTE_SOLVER = [
    {"ORDEM": 10, "MESMA_REG": "SIM", "BUCKET": "ES", "HR_COL": "ES_HR", "PCS_COL": "ES_PCS", "LTP_COL": "LTP_EST_SEG_PCS"},
    {"ORDEM": 20, "MESMA_REG": "NAO", "BUCKET": "PV_PROX", "HR_COL": "PV_PROX_HR", "PCS_COL": "PV_PROX_PCS", "LTP_COL": "LTP_SALDO_PREV_PROX_MES_PCS"},
    {"ORDEM": 30, "MESMA_REG": "NAO", "BUCKET": "ES", "HR_COL": "ES_HR", "PCS_COL": "ES_PCS", "LTP_COL": "LTP_EST_SEG_PCS"},
    {"ORDEM": 40, "MESMA_REG": "SIM", "BUCKET": "PV", "HR_COL": "PV_HR", "PCS_COL": "PV_PCS", "LTP_COL": "LTP_SALDO_PREV_PCS"},
    {"ORDEM": 50, "MESMA_REG": "NAO", "BUCKET": "PV", "HR_COL": "PV_HR", "PCS_COL": "PV_PCS", "LTP_COL": "LTP_SALDO_PREV_PCS"},
    {"ORDEM": 60, "MESMA_REG": "SIM", "BUCKET": "C_AT", "HR_COL": "C_AT_HR", "PCS_COL": "C_AT_PCS", "LTP_COL": "LTP_CART_MES_ATUAL"},
    {"ORDEM": 70, "MESMA_REG": "NAO", "BUCKET": "C_AT", "HR_COL": "C_AT_HR", "PCS_COL": "C_AT_PCS", "LTP_COL": "LTP_CART_MES_ATUAL"},
    {"ORDEM": 80, "MESMA_REG": "SIM", "BUCKET": "C_ARR", "HR_COL": "C_ARR_HR", "PCS_COL": "C_ARR_PCS", "LTP_COL": "LTP_CART_ARR_MES_ANT"},
    {"ORDEM": 90, "MESMA_REG": "NAO", "BUCKET": "C_ARR", "HR_COL": "C_ARR_HR", "PCS_COL": "C_ARR_PCS", "LTP_COL": "LTP_CART_ARR_MES_ANT"},
]

LTP_COLS_CORTAVEIS = sorted({b["LTP_COL"] for b in BUCKETS_CORTE_SOLVER})


# -----------------------------
# 8.1 Helpers gerais
# -----------------------------
def _num(series, default=0.0):
    """Converte uma coluna para numero e substitui valores invalidos por default."""
    return pd.to_numeric(series, errors="coerce").fillna(default)


def _normalizar_texto(series):
    """Padroniza texto para comparacoes de chaves e flags."""
    return series.astype(str).str.strip().str.upper()


def _obter_config_dimensao(nome_corte):
    """Traduz RECURSO/FERRAMENTA do painel para colunas e funcoes usadas pelo solver."""
    nome = str(nome_corte).strip().upper()
    if nome == "RECURSO":
        return {
            "NOME": "RECURSO",
            "DIMENSAO_LOG": "REC",
            "COL_CHAVE": "ALOC_REC",
            "COL_CAP": "HOR_REC",
            "COL_ESTOURO_HR": "NEC_ESTOURO_HR_REC",
            "FN_MATRIZ": cria_bd_mat_cortes_REC,
        }
    if nome == "FERRAMENTA":
        return {
            "NOME": "FERRAMENTA",
            "DIMENSAO_LOG": "FER",
            "COL_CHAVE": "COD_FER_UNID",
            "COL_CAP": "HOR_FER",
            "COL_ESTOURO_HR": "NEC_ESTOURO_HR_FER",
            "FN_MATRIZ": cria_bd_mat_cortes_FER,
        }
    raise ValueError(f"Dimensao de corte invalida: {nome_corte}")


# -----------------------------
# 8.2 Recalculo do modelo mensal
# -----------------------------
def _recalcular_modelo_mensal(bd_mes):
    """
    Recalcula o modelo completo do mes.

    Este helper concentra o trecho pesado do pipeline:
    - necessidade PCS/HR;
    - distribuicao de capacidade;
    - demais campos de estouro;
    - explosao de estrutura;
    - necessidade liquida de componentes;
    - decomposicao de NEC_PCS em buckets cortaveis.

    O solver chama este bloco apos aplicar cortes para medir o novo estado real do M1.
    """
    bd_mes = calc_nec_pcs_hr(bd_mes, lote_min_flag, multiplo_emb_flag)
    bd_mes, tab_HOR_REC, tab_HOR_FER = calcular_distrib_capacidade(bd_mes, lote_min_flag, multiplo_emb_flag)
    bd_mes = calcular_demais_campos(bd_mes)

    bd_ltp_estrutura_explodida = explodir_estrutura_ltp(bd_estrutura_filtrada, bd_mes)

    if bd_ltp_estrutura_explodida.empty:
        bd_nec_comp_expl = pd.DataFrame(columns=["UNID_PROD", "COD_INSUMO", "NEC_LIQ_PCS"])
    else:
        bd_nec_comp_expl = calcular_explosao_necessidades(
            bd_ltp_estrutura_explodida,
            bd_mes,
            lote_min_flag,
            multiplo_emb_flag,
        )

    bd_mes = atualizar_ltp_comp_nec_pcs(bd_mes, bd_nec_comp_expl)

    bd_mes = calc_nec_pcs_hr(bd_mes, lote_min_flag, multiplo_emb_flag)
    bd_mes, tab_HOR_REC, tab_HOR_FER = calcular_distrib_capacidade(bd_mes, lote_min_flag, multiplo_emb_flag)
    bd_mes = calcular_demais_campos(bd_mes)
    bd_mes = calcular_decomposicao_nec_pcs_para_precisao_corte(bd_mes)

    return bd_mes, bd_ltp_estrutura_explodida, bd_nec_comp_expl, tab_HOR_REC, tab_HOR_FER


# -----------------------------
# 8.3 Matriz de gargalos
# -----------------------------
def _matriz_cortes_dimensao(bd_mes, cfg_dim):
    """Monta os gargalos positivos da dimensao ativa, ja aplicando filtros de debug."""
    mat = cfg_dim["FN_MATRIZ"](bd_mes).copy()
    mat["CORTE_HR"] = _num(mat["CORTE_HR"])
    mat = mat.loc[mat["CORTE_HR"] > SOLVER_TOL_HR].reset_index(drop=True)

    if DEBUG_RECURSO and cfg_dim["COL_CHAVE"] == "ALOC_REC":
        mat = mat.loc[mat[cfg_dim["COL_CHAVE"]].astype(str).eq(str(DEBUG_RECURSO))].reset_index(drop=True)
    if DEBUG_FERRAMENTA and cfg_dim["COL_CHAVE"] == "COD_FER_UNID":
        mat = mat.loc[mat[cfg_dim["COL_CHAVE"]].astype(str).eq(str(DEBUG_FERRAMENTA))].reset_index(drop=True)

    return mat.sort_values("CORTE_HR", ascending=False).reset_index(drop=True)


def _montar_saldos_origem_pcs(bd_mes):
    """
    Cria um controle de saldo por demanda origem e coluna LTP.

    A chave e (ID_PROD_UNID_FAT, LTP_COL). Isso evita cortar duas vezes a mesma
    demanda quando o produto aparece em varios recursos/ferramentas.
    """
    saldos = {}
    base = bd_mes[["ID_PROD_UNID_FAT"] + LTP_COLS_CORTAVEIS].copy()
    for col in LTP_COLS_CORTAVEIS:
        tmp = (
            base.groupby("ID_PROD_UNID_FAT", dropna=False)[col]
            .max()
            .reset_index()
        )
        for row in tmp.itertuples(index=False):
            saldos[(row.ID_PROD_UNID_FAT, col)] = float(getattr(row, col) or 0.0)
    return saldos


def _linhas_unicas_por_demanda(df, ltp_col):
    """Escolhe uma linha representante para cada demanda origem antes de montar a bd_ECO."""
    if df.empty:
        return df.copy()
    ordem_cols = [c for c in ["PRIOR_MATPAR", "PRIOR_ROT", "IND"] if c in df.columns]
    if ordem_cols:
        df = df.sort_values(ordem_cols).copy()
    return df.drop_duplicates(subset=["ID_PROD_UNID_FAT", ltp_col], keep="first").reset_index(drop=True)


# -----------------------------
# 8.4 bd_ECO de corte direto
# -----------------------------
def _montar_bd_eco_direto(sub_gargalo, bucket):
    """
    Monta a bd_ECO quando o produto que causa o gargalo pode ser cortado diretamente.

    Exemplo: um PA/MR tem saldo em ES/PV/CARTEIRA dentro do proprio recurso ou ferramenta.
    Neste caso, cortar PCS na coluna LTP reduz horas do mesmo item.
    """
    hr_col = bucket["HR_COL"]
    ltp_col = bucket["LTP_COL"]
    pcs_col = bucket["PCS_COL"]

    mask = (
        sub_gargalo["MESMA_REG"].eq(bucket["MESMA_REG"]) &
        _num(sub_gargalo[hr_col]).gt(SOLVER_TOL_HR) &
        _num(sub_gargalo["PCS_HORA"]).gt(0) &
        sub_gargalo["TIPO_PROD"].astype(str).str.upper().ne("PI")
    )

    bd_ECO_DIR = sub_gargalo.loc[mask].copy()
    if DEBUG_PRODUTO:
        bd_ECO_DIR = bd_ECO_DIR.loc[bd_ECO_DIR["COD_PROD"].astype(str).eq(str(DEBUG_PRODUTO))].copy()
    if bd_ECO_DIR.empty:
        return pd.DataFrame()

    bd_ECO_DIR = _linhas_unicas_por_demanda(bd_ECO_DIR, ltp_col)
    bd_ECO_DIR["SOURCE_KIND"] = "DIRETO"
    bd_ECO_DIR["SOURCE_ID"] = bd_ECO_DIR["ID_PROD_UNID_FAT"]
    bd_ECO_DIR["SOURCE_IND"] = bd_ECO_DIR["IND"]
    bd_ECO_DIR["SOURCE_COD_PROD"] = bd_ECO_DIR["COD_PROD"]
    bd_ECO_DIR["SOURCE_UNID_FAT"] = bd_ECO_DIR["UNID_FAT"]
    bd_ECO_DIR["SOURCE_UNID_PROD"] = bd_ECO_DIR["UNID_PROD"]
    bd_ECO_DIR["SOURCE_LTP_COL"] = ltp_col
    bd_ECO_DIR["SOURCE_BUCKET"] = bucket["BUCKET"]
    bd_ECO_DIR["TARGET_IND"] = bd_ECO_DIR["IND"]
    bd_ECO_DIR["TARGET_COD_PROD"] = bd_ECO_DIR["COD_PROD"]
    bd_ECO_DIR["TARGET_TIPO_PROD"] = bd_ECO_DIR["TIPO_PROD"]
    bd_ECO_DIR["TARGET_KEY"] = bd_ECO_DIR["IND"].astype(str) + "|" + hr_col + "|DIRETO"
    bd_ECO_DIR["ORDEM_BUCKET"] = bucket["ORDEM"]
    bd_ECO_DIR["MESMA_REG_ALVO"] = bucket["MESMA_REG"]
    bd_ECO_DIR["COL_HR_ALVO"] = hr_col
    bd_ECO_DIR["COL_LTP_ALVO"] = ltp_col
    bd_ECO_DIR["PCS_POR_HR_ALIVIO"] = _num(bd_ECO_DIR["PCS_HORA"])
    bd_ECO_DIR["TARGET_HR_DISP_CALC"] = _num(bd_ECO_DIR[hr_col]).clip(lower=0)
    bd_ECO_DIR["SOURCE_PCS_DISP_CALC"] = np.minimum(
        _num(bd_ECO_DIR[ltp_col]).clip(lower=0),
        _num(bd_ECO_DIR[pcs_col]).clip(lower=0),
    )
    bd_ECO_DIR["HR_DISP_CALC"] = np.where(
        bd_ECO_DIR["PCS_POR_HR_ALIVIO"] > 0,
        bd_ECO_DIR["SOURCE_PCS_DISP_CALC"] / bd_ECO_DIR["PCS_POR_HR_ALIVIO"],
        0.0,
    )

    keep_cols = [
        "SOURCE_KIND", "SOURCE_ID", "SOURCE_IND", "SOURCE_COD_PROD", "SOURCE_UNID_FAT", "SOURCE_UNID_PROD",
        "SOURCE_LTP_COL", "SOURCE_BUCKET", "TARGET_IND", "TARGET_COD_PROD", "TARGET_TIPO_PROD", "TARGET_KEY",
        "ORDEM_BUCKET", "MESMA_REG_ALVO", "COL_HR_ALVO", "COL_LTP_ALVO", "PCS_POR_HR_ALIVIO",
        "TARGET_HR_DISP_CALC", "SOURCE_PCS_DISP_CALC", "HR_DISP_CALC", "PRIOR_MATPAR", "PRIOR_ROT", "IND",
    ]
    return bd_ECO_DIR[keep_cols].reset_index(drop=True)


# -----------------------------
# 8.5 bd_ECO PA -> PI
# -----------------------------
def _montar_bd_eco_pi(sub_gargalo, bd_mes, bucket):
    """
    Monta a bd_ECO para o caso especial de PI.

    Quando o gargalo esta em um PI, a demanda dele pode ter sido gerada por PAs.
    O solver procura na estrutura quais PAs consomem esse PI e cria linhas da bd_ECO de
    corte na origem PA. Assim o corte reduz a demanda derivada do PI em vez de
    cortar cegamente o proprio PI.
    """
    hr_col = bucket["HR_COL"]
    ltp_col = bucket["LTP_COL"]
    pcs_col = bucket["PCS_COL"]

    pi_rows = sub_gargalo.loc[
        sub_gargalo["TIPO_PROD"].astype(str).str.upper().eq("PI") &
        sub_gargalo["MESMA_REG"].eq(bucket["MESMA_REG"]) &
        _num(sub_gargalo[hr_col]).gt(SOLVER_TOL_HR) &
        _num(sub_gargalo["PCS_HORA"]).gt(0)
    ].copy()

    if pi_rows.empty or bd_estr_fator_estrutural.empty:
        return pd.DataFrame()

    lista_bd_ECO_PI = []
    fator_cols = ["UNID_PROD", "COD_PROD", "COD_INSUMO", "FATOR_ESTRUTURAL"]
    fatores = bd_estr_fator_estrutural[fator_cols].copy()
    fatores["COD_PROD"] = fatores["COD_PROD"].astype(str)
    fatores["COD_INSUMO"] = fatores["COD_INSUMO"].astype(str)
    fatores["UNID_PROD"] = fatores["UNID_PROD"].astype(str)

    for pi in pi_rows.itertuples(index=False):
        cod_pi = str(pi.COD_PROD)
        unid_pi = str(pi.UNID_PROD)
        pcs_hora_pi = float(getattr(pi, "PCS_HORA") or 0.0)
        target_hr_disp = float(getattr(pi, hr_col) or 0.0)

        links = fatores.loc[
            fatores["UNID_PROD"].eq(unid_pi) &
            fatores["COD_INSUMO"].eq(cod_pi) &
            _num(fatores["FATOR_ESTRUTURAL"]).gt(0)
        ].copy()
        if links.empty or pcs_hora_pi <= 0 or target_hr_disp <= SOLVER_TOL_HR:
            continue

        pais = bd_mes.loc[
            bd_mes["UNID_PROD"].astype(str).eq(unid_pi) &
            bd_mes["COD_PROD"].astype(str).isin(set(links["COD_PROD"])) &
            bd_mes["TIPO_PROD"].astype(str).str.upper().ne("PI") &
            _num(bd_mes[hr_col]).gt(SOLVER_TOL_HR)
        ].copy()
        if DEBUG_PRODUTO:
            pais = pais.loc[pais["COD_PROD"].astype(str).eq(str(DEBUG_PRODUTO))].copy()
        if pais.empty:
            continue

        pais = pais.merge(
            links[["UNID_PROD", "COD_PROD", "COD_INSUMO", "FATOR_ESTRUTURAL"]],
            how="inner",
            on=["UNID_PROD", "COD_PROD"],
        )
        if pais.empty:
            continue

        pais = _linhas_unicas_por_demanda(pais, ltp_col)
        pais["SOURCE_KIND"] = "PI_TRACE_PA"
        pais["SOURCE_ID"] = pais["ID_PROD_UNID_FAT"]
        pais["SOURCE_IND"] = pais["IND"]
        pais["SOURCE_COD_PROD"] = pais["COD_PROD"]
        pais["SOURCE_UNID_FAT"] = pais["UNID_FAT"]
        pais["SOURCE_UNID_PROD"] = pais["UNID_PROD"]
        pais["SOURCE_LTP_COL"] = ltp_col
        pais["SOURCE_BUCKET"] = bucket["BUCKET"]
        pais["TARGET_IND"] = getattr(pi, "IND")
        pais["TARGET_COD_PROD"] = cod_pi
        pais["TARGET_TIPO_PROD"] = "PI"
        pais["TARGET_KEY"] = str(getattr(pi, "IND")) + "|" + hr_col + "|PI_TRACE"
        pais["ORDEM_BUCKET"] = bucket["ORDEM"]
        pais["MESMA_REG_ALVO"] = bucket["MESMA_REG"]
        pais["COL_HR_ALVO"] = hr_col
        pais["COL_LTP_ALVO"] = ltp_col
        pais["PCS_POR_HR_ALIVIO"] = pcs_hora_pi / _num(pais["FATOR_ESTRUTURAL"]).replace(0, np.nan)
        pais["PCS_POR_HR_ALIVIO"] = pais["PCS_POR_HR_ALIVIO"].replace([np.inf, -np.inf], np.nan).fillna(0)
        pais["TARGET_HR_DISP_CALC"] = target_hr_disp
        pais["SOURCE_PCS_DISP_CALC"] = np.minimum(
            _num(pais[ltp_col]).clip(lower=0),
            _num(pais[pcs_col]).clip(lower=0),
        )
        pais["HR_DISP_CALC"] = np.where(
            pais["PCS_POR_HR_ALIVIO"] > 0,
            pais["SOURCE_PCS_DISP_CALC"] / pais["PCS_POR_HR_ALIVIO"],
            0.0,
        )
        pais["RASTREABILIDADE_PI"] = pais["SOURCE_COD_PROD"].astype(str) + "->" + cod_pi
        lista_bd_ECO_PI.append(pais)

    if not lista_bd_ECO_PI:
        return pd.DataFrame()

    bd_ECO_PI = pd.concat(lista_bd_ECO_PI, ignore_index=True)
    keep_cols = [
        "SOURCE_KIND", "SOURCE_ID", "SOURCE_IND", "SOURCE_COD_PROD", "SOURCE_UNID_FAT", "SOURCE_UNID_PROD",
        "SOURCE_LTP_COL", "SOURCE_BUCKET", "TARGET_IND", "TARGET_COD_PROD", "TARGET_TIPO_PROD", "TARGET_KEY",
        "ORDEM_BUCKET", "MESMA_REG_ALVO", "COL_HR_ALVO", "COL_LTP_ALVO", "PCS_POR_HR_ALIVIO",
        "TARGET_HR_DISP_CALC", "SOURCE_PCS_DISP_CALC", "HR_DISP_CALC", "PRIOR_MATPAR", "PRIOR_ROT", "IND",
        "COD_INSUMO", "FATOR_ESTRUTURAL", "RASTREABILIDADE_PI",
    ]
    return bd_ECO_PI[keep_cols].reset_index(drop=True)


# -----------------------------
# 8.6 Montagem da bd_ECO por gargalo
# -----------------------------
def _montar_bd_eco_gargalo(bd_mes, cfg_dim, row_gargalo):
    """Une bd_ECO direta e bd_ECO PA -> PI para um recurso/ferramenta especifico."""
    col_chave = cfg_dim["COL_CHAVE"]
    chave_dim = row_gargalo[col_chave]
    unid_prod = row_gargalo["UNID_PROD"]

    sub = bd_mes.loc[
        bd_mes["UNID_PROD"].astype(str).eq(str(unid_prod)) &
        bd_mes[col_chave].astype(str).eq(str(chave_dim))
    ].copy()

    if sub.empty:
        return pd.DataFrame()

    partes = []
    for bucket in BUCKETS_CORTE_SOLVER:
        partes.append(_montar_bd_eco_direto(sub, bucket))
        partes.append(_montar_bd_eco_pi(sub, bd_mes, bucket))

    partes = [p for p in partes if p is not None and not p.empty]
    if not partes:
        return pd.DataFrame()

    bd_ECO_CORTE = pd.concat(partes, ignore_index=True)
    bd_ECO_CORTE = bd_ECO_CORTE.loc[
        _num(bd_ECO_CORTE["PCS_POR_HR_ALIVIO"]).gt(0) &
        _num(bd_ECO_CORTE["HR_DISP_CALC"]).gt(SOLVER_TOL_HR)
    ].copy()

    if bd_ECO_CORTE.empty:
        return pd.DataFrame()

    bd_ECO_CORTE = bd_ECO_CORTE.sort_values(
        by=["ORDEM_BUCKET", "PRIOR_MATPAR", "PRIOR_ROT", "IND"],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    bd_ECO_CORTE["DIMENSAO_CORTE"] = cfg_dim["DIMENSAO_LOG"]
    bd_ECO_CORTE["CAMPO_DIMENSAO_CORTE"] = col_chave
    bd_ECO_CORTE["VALOR_DIMENSAO_CORTE"] = chave_dim
    bd_ECO_CORTE["UNID_PROD_EVENTO"] = unid_prod
    return bd_ECO_CORTE


# -----------------------------
# 8.7 Solver de uma dimensao
# -----------------------------
def _resolver_dimensao_solver(bd_mes, nome_corte, ciclo_solver, logs_buffer, resumo_buffer):
    """
    Resolve uma fase de corte: RECURSO ou FERRAMENTA.

    Para cada gargalo positivo, o solver percorre a bd_ECO na ordem de prioridade
    dos buckets. Ele controla saldo por origem para impedir duplo corte quando uma
    mesma demanda aparece em mais de uma rota/recurso/ferramenta.
    """
    cfg_dim = _obter_config_dimensao(nome_corte)
    mat_cortes = _matriz_cortes_dimensao(bd_mes, cfg_dim)

    if mat_cortes.empty:
        return bd_mes, 0.0

    saldos_origem_pcs = _montar_saldos_origem_pcs(bd_mes)
    saldos_target_hr = {}
    cortes_eventos = []
    total_hr_cortada_dim = 0.0

    for seq_gargalo, row in enumerate(mat_cortes.itertuples(index=False), start=1):
        row_gargalo = row._asdict()
        objetivo_hr_ini = float(row_gargalo.get("CORTE_HR", 0.0) or 0.0)
        objetivo_hr_restante = objetivo_hr_ini
        bd_ECO_CORTE = _montar_bd_eco_gargalo(bd_mes, cfg_dim, row_gargalo)
        qtd_linhas_bd_ECO = int(len(bd_ECO_CORTE))
        hr_cortada_gargalo = 0.0
        pcs_cortada_gargalo = 0.0

        if bd_ECO_CORTE.empty:
            resumo_buffer.append({
                "CICLO_SOLVER": ciclo_solver,
                "DIMENSAO_CORTE": cfg_dim["DIMENSAO_LOG"],
                "CAMPO_DIMENSAO_CORTE": cfg_dim["COL_CHAVE"],
                "VALOR_DIMENSAO_CORTE": row_gargalo[cfg_dim["COL_CHAVE"]],
                "UNID_PROD_EVENTO": row_gargalo["UNID_PROD"],
                "SEQ_GARGALO": seq_gargalo,
                "OBJETIVO_HR_INI": objetivo_hr_ini,
                "HORAS_CORTADAS": 0.0,
                "OBJETIVO_HR_RESIDUAL": objetivo_hr_restante,
                "QTD_LINHAS_BD_ECO": 0,
                "STATUS_GARGALO": "SEM_BD_ECO_CORTAVEL",
            })
            continue

        # Percorre a bd_ECO em ordem de prioridade. O loop para assim que o
        # gargalo atual estiver fechado ou quando nao houver saldo cortavel.
        for linha_eco in bd_ECO_CORTE.itertuples(index=False):
            if objetivo_hr_restante <= SOLVER_TOL_HR:
                break

            # Saldo da origem: impede que um mesmo PA/bucket seja cortado duas vezes
            # quando ele alivia mais de um gargalo.
            source_key = (linha_eco.SOURCE_ID, linha_eco.SOURCE_LTP_COL)
            source_pcs_rem = float(saldos_origem_pcs.get(source_key, 0.0) or 0.0)
            if source_pcs_rem <= SOLVER_TOL_HR:
                continue

            pcs_por_hr = float(linha_eco.PCS_POR_HR_ALIVIO or 0.0)
            if pcs_por_hr <= 0:
                continue

            # Saldo do alvo: impede cortar mais horas do que o item/bucket possui
            # dentro do gargalo que esta sendo resolvido.
            target_key = linha_eco.TARGET_KEY
            target_hr_rem = float(saldos_target_hr.get(target_key, linha_eco.TARGET_HR_DISP_CALC) or 0.0)
            if target_hr_rem <= SOLVER_TOL_HR:
                continue

            # Corte preciso: sempre usa o minimo entre objetivo restante, saldo da
            # origem, saldo do alvo e disponibilidade calculada da linha da bd_ECO.
            hr_disp_source = source_pcs_rem / pcs_por_hr
            hr_disp_calc = float(linha_eco.HR_DISP_CALC or 0.0)
            hr_corte = min(objetivo_hr_restante, hr_disp_source, target_hr_rem, hr_disp_calc)
            if hr_corte <= SOLVER_TOL_HR:
                continue

            pcs_corte = hr_corte * pcs_por_hr
            saldos_origem_pcs[source_key] = max(source_pcs_rem - pcs_corte, 0.0)
            saldos_target_hr[target_key] = max(target_hr_rem - hr_corte, 0.0)
            objetivo_hr_restante = max(objetivo_hr_restante - hr_corte, 0.0)
            hr_cortada_gargalo += hr_corte
            pcs_cortada_gargalo += pcs_corte
            total_hr_cortada_dim += hr_corte

            cortes_eventos.append({
                "CICLO_SOLVER": ciclo_solver,
                "DIMENSAO_CORTE": cfg_dim["DIMENSAO_LOG"],
                "CAMPO_DIMENSAO_CORTE": cfg_dim["COL_CHAVE"],
                "VALOR_DIMENSAO_CORTE": row_gargalo[cfg_dim["COL_CHAVE"]],
                "UNID_PROD_EVENTO": row_gargalo["UNID_PROD"],
                "SEQ_GARGALO": seq_gargalo,
                "OBJETIVO_HR_INI": objetivo_hr_ini,
                "OBJETIVO_HR_RESTANTE_ANTES": objetivo_hr_restante + hr_corte,
                "OBJETIVO_HR_RESTANTE_DEPOIS": objetivo_hr_restante,
                "SOURCE_KIND": linha_eco.SOURCE_KIND,
                "SOURCE_ID": linha_eco.SOURCE_ID,
                "SOURCE_IND": linha_eco.SOURCE_IND,
                "SOURCE_COD_PROD": linha_eco.SOURCE_COD_PROD,
                "SOURCE_UNID_FAT": linha_eco.SOURCE_UNID_FAT,
                "SOURCE_UNID_PROD": linha_eco.SOURCE_UNID_PROD,
                "TARGET_IND": linha_eco.TARGET_IND,
                "TARGET_COD_PROD": linha_eco.TARGET_COD_PROD,
                "TARGET_TIPO_PROD": linha_eco.TARGET_TIPO_PROD,
                "MESMA_REG_ALVO": linha_eco.MESMA_REG_ALVO,
                "BUCKET": linha_eco.SOURCE_BUCKET,
                "COL_HR_ALVO": linha_eco.COL_HR_ALVO,
                "COL_LTP_ALVO": linha_eco.COL_LTP_ALVO,
                "PCS_POR_HR_ALIVIO": pcs_por_hr,
                "HORAS_CORTADAS_EVENTO": hr_corte,
                "PCS_CORTADAS_EVENTO": pcs_corte,
                "SOURCE_PCS_SALDO_POS_CORTE": saldos_origem_pcs[source_key],
                "TARGET_HR_SALDO_POS_CORTE": saldos_target_hr[target_key],
                "RASTREABILIDADE_PI": getattr(linha_eco, "RASTREABILIDADE_PI", np.nan),
                "FATOR_ESTRUTURAL": getattr(linha_eco, "FATOR_ESTRUTURAL", np.nan),
            })

        status = "FECHADO_NO_LOTE" if objetivo_hr_restante <= SOLVER_TOL_HR else "RESIDUO_APOS_BD_ECO"
        resumo_buffer.append({
            "CICLO_SOLVER": ciclo_solver,
            "DIMENSAO_CORTE": cfg_dim["DIMENSAO_LOG"],
            "CAMPO_DIMENSAO_CORTE": cfg_dim["COL_CHAVE"],
            "VALOR_DIMENSAO_CORTE": row_gargalo[cfg_dim["COL_CHAVE"]],
            "UNID_PROD_EVENTO": row_gargalo["UNID_PROD"],
            "SEQ_GARGALO": seq_gargalo,
            "OBJETIVO_HR_INI": objetivo_hr_ini,
            "HORAS_CORTADAS": hr_cortada_gargalo,
            "PCS_CORTADAS": pcs_cortada_gargalo,
            "OBJETIVO_HR_RESIDUAL": objetivo_hr_restante,
            "QTD_LINHAS_BD_ECO": qtd_linhas_bd_ECO,
            "STATUS_GARGALO": status,
        })

    if cortes_eventos:
        df_cortes = pd.DataFrame(cortes_eventos)
        bd_mes = _aplicar_cortes_ltp(bd_mes, df_cortes)
        logs_buffer.extend(cortes_eventos)

    return bd_mes, total_hr_cortada_dim


# -----------------------------
# 8.8 Aplicacao dos cortes
# -----------------------------
def _aplicar_cortes_ltp(bd_mes, df_cortes):
    """
    Aplica os cortes consolidados nas colunas LTP.

    O corte e agregado por SOURCE_ID + COL_LTP_ALVO antes da aplicacao. Isso garante
    que uma origem so perca o total consolidado decidido pelo solver.
    """
    bd_mes = bd_mes.copy()
    if df_cortes.empty:
        return bd_mes

    cortes_agg = (
        df_cortes.groupby(["SOURCE_ID", "COL_LTP_ALVO"], dropna=False)["PCS_CORTADAS_EVENTO"]
        .sum()
        .reset_index()
    )

    for col_ltp in sorted(cortes_agg["COL_LTP_ALVO"].dropna().unique()):
        mapa_cortes = (
            cortes_agg.loc[cortes_agg["COL_LTP_ALVO"].eq(col_ltp)]
            .set_index("SOURCE_ID")["PCS_CORTADAS_EVENTO"]
            .to_dict()
        )
        corte = bd_mes["ID_PROD_UNID_FAT"].map(mapa_cortes).fillna(0.0)
        bd_mes[col_ltp] = (_num(bd_mes[col_ltp]) - corte).clip(lower=0)

    return bd_mes


# -----------------------------
# 8.9 Validacao de fechamento
# -----------------------------
def _resumo_fechamento_dimensoes(bd_mes, ciclo_solver):
    """Recalcula gargalos positivos em RECURSO e FERRAMENTA para medir residuo."""
    partes = []
    for nome_corte in ["RECURSO", "FERRAMENTA"]:
        cfg = _obter_config_dimensao(nome_corte)
        mat = cfg["FN_MATRIZ"](bd_mes).copy()
        mat["CORTE_HR"] = _num(mat["CORTE_HR"])
        mat = mat.loc[mat["CORTE_HR"] > SOLVER_TOL_HR].copy()
        if mat.empty:
            continue
        mat["CICLO_SOLVER"] = ciclo_solver
        mat["DIMENSAO_CORTE"] = cfg["DIMENSAO_LOG"]
        mat["CAMPO_DIMENSAO_CORTE"] = cfg["COL_CHAVE"]
        partes.append(mat)
    if partes:
        return pd.concat(partes, ignore_index=True)
    return pd.DataFrame()


# -----------------------------
# 8.9.1 Exportacao Excel segura
# -----------------------------
def _exportar_excel_seguro(df, caminho, sheet_base="log"):
    """
    Exporta DataFrame para Excel.

    Se o log passar do limite de linhas do Excel, a funcao divide o arquivo
    em varias abas. Isso evita falha na hora de gerar o material de validacao
    para o usuario.
    """
    if df.empty:
        return

    caminho = Path(caminho)
    max_linhas_excel = 1_048_000
    sheet_base = str(sheet_base)[:25] or "log"

    if len(df) <= max_linhas_excel:
        df.to_excel(caminho, index=False, sheet_name=sheet_base[:31])
        return

    with pd.ExcelWriter(caminho, engine="openpyxl") as writer:
        for idx, inicio in enumerate(range(0, len(df), max_linhas_excel), start=1):
            fim = inicio + max_linhas_excel
            aba = f"{sheet_base}_{idx}"[:31]
            df.iloc[inicio:fim].to_excel(writer, index=False, sheet_name=aba)


# -----------------------------
# 8.10 Execucao sequencial mensal
# -----------------------------
# A execucao abaixo e a parte principal da celula:
# - normaliza os meses;
# - seleciona M1;
# - calcula o snapshot antes do corte;
# - roda ciclos macro de RECURSO/FERRAMENTA;
# - recalcula o modelo apos cada dimensao cortada;
# - para quando fecha M1 ou quando nao existe mais saldo cortavel.
bd_LTP_NEC = bd_LTP_NEC.copy()
bd_LTP_NEC["MES_REF"] = pd.to_datetime(bd_LTP_NEC["MES_REF"], errors="coerce").dt.normalize()
mes_refs = [pd.Timestamp(x) for x in sorted(bd_LTP_NEC["MES_REF"].dropna().unique().tolist())]

logs_cortes_solver = []
resumo_gargalos_solver = []
resumo_fechamento_solver = []

bd_LTP_MES = pd.DataFrame()
bd_LTP_MES_snapshot = pd.DataFrame()
bd_LTP_MES_DIM = pd.DataFrame()
bd_ltp_estrutura_explodida = pd.DataFrame()
bd_nec_comp_expl = pd.DataFrame()

for mes_ref in mes_refs:

    # Regra intencional: fechar completamente M1 antes de liberar M2, M3...
    if mes_ref != mes_refs[0]:
        continue

    if DEBUG_SOLVER:
        print(f"Iniciando solver de cortes para M1: {mes_ref.strftime('%Y-%m')}")
        print(f"Sequencia de corte: {SEQUENCIA_CORTES_SOLVER}")

    # M1 nasce da base completa bd_LTP_NEC. A ETL anterior continua sendo a fonte da verdade.
    bd_LTP_MES = bd_LTP_NEC.loc[bd_LTP_NEC["MES_REF"].eq(mes_ref)].copy().reset_index(drop=True)

    # Primeiro recalculo cria o estado inicial do mes antes de qualquer corte solver.
    bd_LTP_MES, bd_ltp_estrutura_explodida, bd_nec_comp_expl, tab_HOR_REC, tab_HOR_FER = _recalcular_modelo_mensal(bd_LTP_MES)
    bd_LTP_MES_snapshot = bd_LTP_MES.copy(deep=True)

    # Ciclos macro: cada ciclo tenta resolver a sequencia completa definida no painel.
    # Depois de cada dimensao com corte aplicado, o modelo e recalculado para refletir
    # impactos em PAs, PIs, recursos, ferramentas e componentes.
    for ciclo_solver in range(1, SOLVER_MAX_CICLOS_MES + 1):
        horas_cortadas_ciclo = 0.0

        for nome_corte in SEQUENCIA_CORTES_SOLVER:
            bd_LTP_MES, horas_cortadas_dim = _resolver_dimensao_solver(
                bd_LTP_MES,
                nome_corte,
                ciclo_solver,
                logs_cortes_solver,
                resumo_gargalos_solver,
            )
            horas_cortadas_ciclo += horas_cortadas_dim

            if horas_cortadas_dim > SOLVER_TOL_HR:
                bd_LTP_MES, bd_ltp_estrutura_explodida, bd_nec_comp_expl, tab_HOR_REC, tab_HOR_FER = _recalcular_modelo_mensal(bd_LTP_MES)

        resumo_residuo_ciclo = _resumo_fechamento_dimensoes(bd_LTP_MES, ciclo_solver)
        if not resumo_residuo_ciclo.empty:
            resumo_fechamento_solver.append(resumo_residuo_ciclo)
            residuo_total_hr = float(resumo_residuo_ciclo["CORTE_HR"].sum())
            qtd_gargalos = int(len(resumo_residuo_ciclo))
        else:
            residuo_total_hr = 0.0
            qtd_gargalos = 0

        if DEBUG_SOLVER:
            print(
                f"Ciclo {ciclo_solver}: horas cortadas={horas_cortadas_ciclo:,.2f} | "
                f"residuo={residuo_total_hr:,.2f} h | gargalos={qtd_gargalos}"
            )

        if residuo_total_hr <= SOLVER_TOL_HR:
            if DEBUG_SOLVER:
                print("M1 fechado pelo solver.")
            break

        if horas_cortadas_ciclo <= SOLVER_TOL_HR:
            if DEBUG_SOLVER:
                print("Solver parou: nao ha saldo cortavel adicional nos buckets configurados.")
            break

# -----------------------------
# 8.11 Logs e tabelas de debug
# -----------------------------
# df_log_cortes: log detalhado de cada decisao de corte.
# df_resumo_gargalos_solver: resumo por gargalo processado.
# df_resumo_fechamento: gargalos restantes apos cada ciclo.
# bd_LTP_MES_DIM: subconjunto de linhas envolvidas em corte para inspecao rapida.
df_log_cortes = pd.DataFrame(logs_cortes_solver)
df_resumo_gargalos_solver = pd.DataFrame(resumo_gargalos_solver)
df_resumo_fechamento = (
    pd.concat(resumo_fechamento_solver, ignore_index=True)
    if resumo_fechamento_solver else pd.DataFrame()
)

if not df_log_cortes.empty and not bd_LTP_MES.empty:
    inds_debug = set(df_log_cortes["SOURCE_IND"].dropna().astype(int).tolist()) | set(df_log_cortes["TARGET_IND"].dropna().astype(int).tolist())
    bd_LTP_MES_DIM = bd_LTP_MES.loc[bd_LTP_MES["IND"].isin(inds_debug)].copy().reset_index(drop=True)
else:
    bd_LTP_MES_DIM = bd_LTP_MES.head(0).copy() if not bd_LTP_MES.empty else pd.DataFrame()

if SOLVER_EXPORTAR_DEBUG and mes_refs:
    mes_ref_arquivo = mes_refs[0].strftime("%Y_%m")

    if not df_log_cortes.empty:
        try:
            df_log_cortes.to_parquet(pasta_output / f"log_cortes_solver_{mes_ref_arquivo}.parquet", index=False)
        except Exception as exc:
            print(f"Nao foi possivel exportar log parquet: {exc}")

        if SOLVER_EXPORTAR_LOG_DETALHADO_XLSX:
            _exportar_excel_seguro(
                df_log_cortes,
                pasta_output / f"log_cortes_solver_{mes_ref_arquivo}.xlsx",
                sheet_base="log_cortes",
            )

    if not df_resumo_gargalos_solver.empty:
        df_resumo_gargalos_solver.to_excel(pasta_output / f"resumo_gargalos_solver_{mes_ref_arquivo}.xlsx", index=False)

    if not df_resumo_fechamento.empty:
        df_resumo_fechamento.to_excel(pasta_output / f"resumo_fechamento_solver_{mes_ref_arquivo}.xlsx", index=False)

if DEBUG_SOLVER:
    total_cortado = float(df_log_cortes["HORAS_CORTADAS_EVENTO"].sum()) if not df_log_cortes.empty else 0.0
    print(f"Solver finalizado. Eventos de corte: {len(df_log_cortes):,} | Horas cortadas: {total_cortado:,.2f}")
    if not df_resumo_fechamento.empty:
        ultimo_ciclo = df_resumo_fechamento["CICLO_SOLVER"].max()
        residuo_final = df_resumo_fechamento.loc[df_resumo_fechamento["CICLO_SOLVER"].eq(ultimo_ciclo), "CORTE_HR"].sum()
        print(f"Residuo final no ultimo ciclo ({ultimo_ciclo}): {residuo_final:,.2f} h")
    else:
        print("Sem residuo final registrado.")


Iniciando solver de cortes para M1: 2026-04
Sequencia de corte: ['RECURSO', 'FERRAMENTA']
Ciclo 1: horas cortadas=79,479.15 | residuo=60,254.00 h | gargalos=254
Ciclo 2: horas cortadas=827.86 | residuo=59,372.17 h | gargalos=236
Ciclo 3: horas cortadas=127.76 | residuo=59,254.58 h | gargalos=231
Ciclo 4: horas cortadas=10.31 | residuo=59,253.22 h | gargalos=230
Ciclo 5: horas cortadas=9.11 | residuo=59,252.07 h | gargalos=230
Ciclo 6: horas cortadas=8.11 | residuo=59,246.48 h | gargalos=229
Solver finalizado. Eventos de corte: 2,300 | Horas cortadas: 80,462.30
Residuo final no ultimo ciclo (6): 59,246.48 h


In [ ]:
# Observacao do motor solver: logs detalhados ficam em df_log_cortes e, quando habilitado, em parquet na pasta output.
# bd_LTP_MES_DIM agora contem as linhas envolvidas em cortes para facilitar debug.


In [9]:
# Exportar para Excel

dataframes_para_exportar = {
    "bd_LTP_MES_snapshot": bd_LTP_MES_snapshot,
    "bd_LTP_MES": bd_LTP_MES,
    "bd_LTP_MES_DIM": bd_LTP_MES_DIM
}

for nome in dataframes_para_exportar:
    df = globals()[nome]
    caminho_arquivo = pasta_output / f"{nome}.xlsx"
    df.to_excel(caminho_arquivo, index=False)
    print(f"✅ Exportado: {caminho_arquivo}")
    
timer.finalizar()
print("🎯 Processo concluído com sucesso!")

✅ Exportado: c:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT\bd_LTP_MES_snapshot.xlsx
✅ Exportado: c:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT\bd_LTP_MES.xlsx
✅ Exportado: c:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT\bd_LTP_MES_DIM.xlsx
Tempo total de processamento: 18 min 26.1 s
🎯 Processo concluído com sucesso!


In [10]:
# CRIAR UM LOG DE CORTES, PARA SABER COMO FOI PROCESSADO E SE FOI PROCESSADO

# Ver quais Bibliotecas devo excluir conforme indicado por Lucas
# Não recomendamos a instalação pois possui vulnerabilidades não corrigidas
# colorama
# numpy
# prompt toolkit
# python-dateutil
